In [1]:
# %pip install seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')
import re
from math import radians, sin, cos, sqrt, atan2

# 设置中文字体显示（如果图表需要显示中文）
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [2]:
# 读取Excel文件
try:
    # 读取租房数据  
    rent_price_df = pd.read_excel(r"D:\desktop\ruc_Class25Q2_test_price.xlsx")
    print("购房数据读取成功！")
    print(f"购房数据形状: {rent_price_df.shape}")
    
except FileNotFoundError as e:
    print(f"文件未找到: {e}")
except Exception as e:
    print(f"读取文件时出错: {e}")

# 查看数据的基本信息
print("\n=== 购房数据基本信息 ===")
print(f"行数: {rent_price_df.shape[0]}, 列数: {rent_price_df.shape[1]}")
print(rent_price_df.info())

# 查看前几行数据
print("\n=== 购房数据前3行 ===")
print(rent_price_df.head(3))

# 检查列名
print("\n=== 购房数据列名 ===")
print(rent_price_df.columns.tolist())

购房数据读取成功！
购房数据形状: (34017, 55)

=== 购房数据基本信息 ===
行数: 34017, 列数: 55
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34017 entries, 0 to 34016
Data columns (total 55 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   ID         34017 non-null  int64         
 1   城市         34017 non-null  int64         
 2   区域         34017 non-null  int64         
 3   板块         34017 non-null  int64         
 4   环线         15677 non-null  object        
 5   房屋户型       34003 non-null  object        
 6   所在楼层       34017 non-null  object        
 7   建筑面积       34017 non-null  object        
 8   套内面积       9915 non-null   object        
 9   房屋朝向       34017 non-null  object        
 10  建筑结构       34003 non-null  object        
 11  装修情况       34003 non-null  object        
 12  梯户比例       33382 non-null  object        
 13  配备电梯       29925 non-null  object        
 14  别墅类型       154 non-null    object        
 15  交易时间       34017 non-

In [3]:
# 2.1 装修变量
# 2.1.1 分布检查
print("装修情况分布:")
print(rent_price_df['装修情况'].value_counts())

# 2.1.2 缺失值检查
print(f"\n装修情况缺失值数量: {rent_price_df['装修情况'].isna().sum()}")

# 2.1.3 创建装修情况哑变量
# 删除已存在的装修哑变量列
decoration_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('decoration_')]
rent_price_df = rent_price_df.drop(columns=decoration_columns_to_drop)

# 定义所有可能的装修类型
all_decoration_types = ['精装', '简装', '毛坯', '其他']

# 创建哑变量
decoration_dummies = pd.get_dummies(rent_price_df['装修情况'], prefix='decoration')

# 确保包含所有装修类型（即使某些类型没有数据）
for deco_type in all_decoration_types:
    col_name = f"decoration_{deco_type}"
    if col_name not in decoration_dummies.columns:
        decoration_dummies[col_name] = 0

# 按正确的顺序排列列
decoration_dummies = decoration_dummies.reindex(columns=[f"decoration_{deco_type}" for deco_type in all_decoration_types])

# 将布尔值转换为整数 (True/False -> 1/0)
decoration_dummies = decoration_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['装修情况'].isna().any():
    na_mask = rent_price_df['装修情况'].isna()
    for col in decoration_dummies.columns:
        decoration_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框
rent_price_df = pd.concat([rent_price_df, decoration_dummies], axis=1)

print(f"\n已创建的装修情况哑变量:")
decoration_dummy_columns = [col for col in rent_price_df.columns if col.startswith('decoration_')]
print(decoration_dummy_columns)

print("\n装修情况哑变量分布:")
for col in decoration_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("装修情况哑变量创建完成（保留缺失值）")

装修情况分布:
装修情况
精装    15971
简装     7988
其他     7134
毛坯     2910
Name: count, dtype: int64

装修情况缺失值数量: 14

已创建的装修情况哑变量:
['decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他']

装修情况哑变量分布:
decoration_精装: 15971个为1, 18032个为0, 14个缺失
decoration_简装: 7988个为1, 26015个为0, 14个缺失
decoration_毛坯: 2910个为1, 31093个为0, 14个缺失
decoration_其他: 7134个为1, 26869个为0, 14个缺失
装修情况哑变量创建完成（保留缺失值）


In [4]:
# 2.2 楼层变量
# 2.2.1 分布检查
print("楼层分布:")
print(rent_price_df['所在楼层'].value_counts())

# 2.2.2 缺失值检查
print(f"\n楼层缺失值数量: {rent_price_df['所在楼层'].isna().sum()}")

# 2.2.3 创建楼层变量
# 删除已存在的楼层变量列
floor_columns_to_drop = [col for col in rent_price_df.columns if 
                        col in ['high_dummy', 'middle_dummy', 'low_dummy', 
                               'basement_dummy', 'top_dummy', 'bottom_dummy', 'total_floor']]
rent_price_df = rent_price_df.drop(columns=floor_columns_to_drop)

# 从楼层数据中提取7个变量:
# 1. 高哑变量 (high_dummy)
# 2. 中哑变量 (middle_dummy) 
# 3. 低哑变量 (low_dummy)
# 4. 地下室哑变量 (basement_dummy)
# 5. 顶层哑变量 (top_dummy)
# 6. 底层哑变量 (bottom_dummy)
# 7. 总楼层连续变量 (total_floor)

def extract_floor_features_purchase(df, floor_column='所在楼层'):
    """
    从购房数据楼层信息中提取7个变量
    """
    # 创建新列，初始化为NaN
    df['high_dummy'] = np.nan
    df['middle_dummy'] = np.nan
    df['low_dummy'] = np.nan
    df['basement_dummy'] = np.nan
    df['top_dummy'] = np.nan
    df['bottom_dummy'] = np.nan
    df['total_floor'] = np.nan
    
    # 只对非空楼层进行处理
    valid_mask = df[floor_column].notna()
    
    # 将楼层列转换为字符串类型
    floor_str = df.loc[valid_mask, floor_column].astype(str)
    
    # 初始化有效数据中的哑变量为0
    df.loc[valid_mask, 'high_dummy'] = 0
    df.loc[valid_mask, 'middle_dummy'] = 0
    df.loc[valid_mask, 'low_dummy'] = 0
    df.loc[valid_mask, 'basement_dummy'] = 0
    df.loc[valid_mask, 'top_dummy'] = 0
    df.loc[valid_mask, 'bottom_dummy'] = 0
    
    # 处理地下室情况
    basement_mask = floor_str.str.contains('地下室')
    df.loc[valid_mask & basement_mask, 'basement_dummy'] = 1
    
    # 处理顶层情况
    top_mask = floor_str.str.contains('顶层')
    df.loc[valid_mask & top_mask, 'top_dummy'] = 1
    
    # 处理底层情况
    bottom_mask = floor_str.str.contains('底层')
    df.loc[valid_mask & bottom_mask, 'bottom_dummy'] = 1
    
    # 处理高楼层情况
    high_mask = floor_str.str.contains('高楼层') & ~top_mask
    df.loc[valid_mask & high_mask, 'high_dummy'] = 1
    
    # 处理中楼层情况
    middle_mask = floor_str.str.contains('中楼层')
    df.loc[valid_mask & middle_mask, 'middle_dummy'] = 1
    
    # 处理低楼层情况
    low_mask = floor_str.str.contains('低楼层') & ~bottom_mask
    df.loc[valid_mask & low_mask, 'low_dummy'] = 1
    
    # 提取总楼层数 - 从"共XX层"格式中提取
    # 匹配"共XX层"模式
    total_floor_match = floor_str.str.extract(r'共(\d+)层')
    df.loc[valid_mask, 'total_floor'] = pd.to_numeric(total_floor_match[0], errors='coerce')
    
    # 对于地下室，如果有总楼层信息，保留它
    # 地下室的总楼层信息通常指的是整栋楼的总楼层
    
    # 确保哑变量是互斥的（一个楼层只能属于一种类型）
    # 优先级：地下室 > 顶层/底层 > 高楼层 > 中楼层 > 低楼层
    
    # 如果同时有地下室和其他类型，保留地下室
    conflict_mask = (df['basement_dummy'] == 1) & (
        (df['high_dummy'] == 1) | (df['middle_dummy'] == 1) | 
        (df['low_dummy'] == 1) | (df['top_dummy'] == 1) | (df['bottom_dummy'] == 1)
    )
    if conflict_mask.any():
        df.loc[conflict_mask, 'high_dummy'] = 0
        df.loc[conflict_mask, 'middle_dummy'] = 0
        df.loc[conflict_mask, 'low_dummy'] = 0
        df.loc[conflict_mask, 'top_dummy'] = 0
        df.loc[conflict_mask, 'bottom_dummy'] = 0
    
    # 如果同时有顶层/底层和其他楼层类型，保留顶层/底层
    conflict_mask = (df['top_dummy'] == 1) & (
        (df['high_dummy'] == 1) | (df['middle_dummy'] == 1) | (df['low_dummy'] == 1)
    )
    if conflict_mask.any():
        df.loc[conflict_mask, 'high_dummy'] = 0
        df.loc[conflict_mask, 'middle_dummy'] = 0
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    conflict_mask = (df['bottom_dummy'] == 1) & (
        (df['high_dummy'] == 1) | (df['middle_dummy'] == 1) | (df['low_dummy'] == 1)
    )
    if conflict_mask.any():
        df.loc[conflict_mask, 'high_dummy'] = 0
        df.loc[conflict_mask, 'middle_dummy'] = 0
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    # 高中低楼层互斥
    conflict_mask = (df['high_dummy'] == 1) & ((df['middle_dummy'] == 1) | (df['low_dummy'] == 1))
    if conflict_mask.any():
        df.loc[conflict_mask, 'middle_dummy'] = 0
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    conflict_mask = (df['middle_dummy'] == 1) & (df['low_dummy'] == 1)
    if conflict_mask.any():
        df.loc[conflict_mask, 'low_dummy'] = 0
    
    return df

# 应用函数提取楼层特征
rent_price_df = extract_floor_features_purchase(rent_price_df, '所在楼层')

print(f"\n已创建的楼层变量:")
print(f"high_dummy: 1表示高楼层，0表示非高楼层，NaN表示缺失")
print(f"middle_dummy: 1表示中楼层，0表示非中楼层，NaN表示缺失")
print(f"low_dummy: 1表示低楼层，0表示非低楼层，NaN表示缺失")
print(f"basement_dummy: 1表示地下室，0表示非地下室，NaN表示缺失")
print(f"top_dummy: 1表示顶层，0表示非顶层，NaN表示缺失")
print(f"bottom_dummy: 1表示底层，0表示非底层，NaN表示缺失")
print(f"total_floor: 总楼层数，NaN表示缺失")

print("\n楼层变量统计:")
floor_features = ['high_dummy', 'middle_dummy', 'low_dummy', 'basement_dummy', 'top_dummy', 'bottom_dummy', 'total_floor']
for feature in floor_features:
    if feature == 'total_floor':
        print(f"{feature}:")
        print(f"  非缺失值数量: {rent_price_df[feature].notna().sum()}")
        if rent_price_df[feature].notna().sum() > 0:
            print(f"  平均值: {rent_price_df[feature].mean():.2f}")
            print(f"  标准差: {rent_price_df[feature].std():.2f}")
            print(f"  最小值: {rent_price_df[feature].min()}")
            print(f"  最大值: {rent_price_df[feature].max()}")
        print(f"  缺失值数量: {rent_price_df[feature].isna().sum()}")
    else:
        print(f"{feature}:")
        print(f"  为1的数量: {(rent_price_df[feature] == 1).sum()}")
        print(f"  为0的数量: {(rent_price_df[feature] == 0).sum()}")
        print(f"  缺失值数量: {rent_price_df[feature].isna().sum()}")

print("楼层变量创建完成（保留缺失值）")

楼层分布:
所在楼层
中楼层 (共6层)     2209
高楼层 (共6层)     2000
低楼层 (共6层)     1470
中楼层 (共18层)     950
高楼层 (共18层)     880
              ... 
高楼层 (共54层)       1
中楼层 (共54层)       1
低楼层 (共53层)       1
高楼层 (共53层)       1
高楼层 (共49层)       1
Name: count, Length: 218, dtype: int64

楼层缺失值数量: 0

已创建的楼层变量:
high_dummy: 1表示高楼层，0表示非高楼层，NaN表示缺失
middle_dummy: 1表示中楼层，0表示非中楼层，NaN表示缺失
low_dummy: 1表示低楼层，0表示非低楼层，NaN表示缺失
basement_dummy: 1表示地下室，0表示非地下室，NaN表示缺失
top_dummy: 1表示顶层，0表示非顶层，NaN表示缺失
bottom_dummy: 1表示底层，0表示非底层，NaN表示缺失
total_floor: 总楼层数，NaN表示缺失

楼层变量统计:
high_dummy:
  为1的数量: 10130
  为0的数量: 23887
  缺失值数量: 0
middle_dummy:
  为1的数量: 13214
  为0的数量: 20803
  缺失值数量: 0
low_dummy:
  为1的数量: 9302
  为0的数量: 24715
  缺失值数量: 0
basement_dummy:
  为1的数量: 28
  为0的数量: 33989
  缺失值数量: 0
top_dummy:
  为1的数量: 720
  为0的数量: 33297
  缺失值数量: 0
bottom_dummy:
  为1的数量: 623
  为0的数量: 33394
  缺失值数量: 0
total_floor:
  非缺失值数量: 34017
  平均值: 17.47
  标准差: 10.80
  最小值: 0.0
  最大值: 58.0
  缺失值数量: 0
楼层变量创建完成（保留缺失值）


In [5]:
# 2.3 面积变量
# 2.3.1 空缺值（在前面的描述性统计中已看出，没有空缺值）
# 2.3.2 变量类型转化
rent_price_df['area'] = rent_price_df['建筑面积'].str.replace('㎡', '').astype(float)
print("面积提取完成!")

面积提取完成!


In [6]:
# 2.4 户型
# 2.4.1 分布检查
print("户型分布:")
print(rent_price_df['房屋户型'].value_counts())

# 2.4.2 缺失值检查
print(f"\n户型缺失值数量: {rent_price_df['房屋户型'].isna().sum()}")

# 2.4.3 创建户型变量
# 删除已存在的户型变量列
layout_columns_to_drop = [col for col in rent_price_df.columns if 
                         col in ['room_count', 'hall_count']]
rent_price_df = rent_price_df.drop(columns=layout_columns_to_drop)

# 从户型数据中提取2个变量:
# 1. 室数量 (room_count)
# 2. 厅数量 (hall_count)
# 保留缺失值为NaN，不进行填充

def extract_room_features(df, layout_column='房屋户型'):
    """
    从户型数据中提取2个变量:
    1. 室数量 (room_count)
    2. 厅数量 (hall_count)
    
    处理逻辑:
    - 提取"室"前面的数字作为室数量
    - 提取"厅"前面的数字作为厅数量  
    - 将"房间"、"居室"视为"室"
    - 无法识别的设为NaN
    """
    # 创建新列，初始化为NaN
    df['room_count'] = np.nan
    df['hall_count'] = np.nan
    
    # 只对非空户型进行处理
    valid_mask = df[layout_column].notna()
    
    # 将户型列转换为字符串类型
    layout_str = df.loc[valid_mask, layout_column].astype(str)
    
    # 提取室数量
    # 匹配"数字+室"、"数字+房间"、"数字+居室"
    room_pattern = r'(\d+)(?:室|房间|居室)'
    room_matches = layout_str.str.extract(room_pattern, expand=False)
    df.loc[valid_mask, 'room_count'] = pd.to_numeric(room_matches, errors='coerce')
    
    # 提取厅数量
    # 匹配"数字+厅"
    hall_pattern = r'(\d+)厅'
    hall_matches = layout_str.str.extract(hall_pattern, expand=False)
    df.loc[valid_mask, 'hall_count'] = pd.to_numeric(hall_matches, errors='coerce')
    
    # 特殊处理：对于"未知室"等情况，设为NaN
    unknown_mask = layout_str.str.contains('未知室')
    df.loc[valid_mask & unknown_mask, 'room_count'] = np.nan
    
    # 特殊处理：对于"车库"等情况，设为NaN
    garage_mask = layout_str.str.contains('车库')
    df.loc[valid_mask & garage_mask, 'room_count'] = np.nan
    df.loc[valid_mask & garage_mask, 'hall_count'] = np.nan
    
    return df

# 应用函数提取户型特征
rent_price_df = extract_room_features(rent_price_df, '房屋户型')


print("\n户型变量统计:")
print("room_count:")
print(f"  非缺失值数量: {rent_price_df['room_count'].notna().sum()}")
print(f"  缺失值数量: {rent_price_df['room_count'].isna().sum()}")
if rent_price_df['room_count'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['room_count'].mean():.2f}")
    print(f"  标准差: {rent_price_df['room_count'].std():.2f}")
    print(f"  最小值: {rent_price_df['room_count'].min()}")
    print(f"  最大值: {rent_price_df['room_count'].max()}")

print("\nhall_count:")
print(f"  非缺失值数量: {rent_price_df['hall_count'].notna().sum()}")
print(f"  缺失值数量: {rent_price_df['hall_count'].isna().sum()}")
if rent_price_df['hall_count'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['hall_count'].mean():.2f}")
    print(f"  标准差: {rent_price_df['hall_count'].std():.2f}")
    print(f"  最小值: {rent_price_df['hall_count'].min()}")
    print(f"  最大值: {rent_price_df['hall_count'].max()}")

print("户型变量创建完成（保留缺失值）")

户型分布:
房屋户型
2室1厅1厨1卫    8263
3室2厅1厨2卫    5437
2室2厅1厨1卫    3945
3室2厅1厨1卫    3674
3室1厅1厨1卫    2756
            ... 
8室3厅1厨3卫       1
5室2厅2厨1卫       1
8室1厅1厨3卫       1
8室2厅1厨4卫       1
4室1厅2厨2卫       1
Name: count, Length: 191, dtype: int64

户型缺失值数量: 14

户型变量统计:
room_count:
  非缺失值数量: 34003
  缺失值数量: 14
  平均值: 2.54
  标准差: 0.88
  最小值: 0.0
  最大值: 8.0

hall_count:
  非缺失值数量: 33853
  缺失值数量: 164
  平均值: 1.49
  标准差: 0.55
  最小值: 0.0
  最大值: 5.0
户型变量创建完成（保留缺失值）


In [7]:
# 2.5 朝向
# 2.5.1 分布检查
print("朝向分布:")
print(rent_price_df['房屋朝向'].value_counts())

# 2.5.2 缺失值检查
print(f"\n朝向缺失值数量: {rent_price_df['房屋朝向'].isna().sum()}")

# 2.5.3 创建朝向哑变量
# 删除已存在的朝向哑变量列
orientation_columns_to_drop = [col for col in rent_price_df.columns if 
                              col in ['south_dummy', 'north_south_dummy']]
rent_price_df = rent_price_df.drop(columns=orientation_columns_to_drop)

# 从朝向数据中提取2个哑变量:
# 1. 朝南哑变量 (south_dummy) - 只要包含"南"字就算朝南
# 2. 南北通透哑变量 (north_south_dummy) - 同时包含"南"和"北"字
# 保留缺失值为NaN，不进行填充

# 初始化哑变量为NaN
rent_price_df['south_dummy'] = np.nan
rent_price_df['north_south_dummy'] = np.nan

# 只对非空朝向进行处理
valid_mask = rent_price_df['房屋朝向'].notna()

# 将朝向列转换为字符串类型
orientation_str = rent_price_df.loc[valid_mask, '房屋朝向'].astype(str)

# 提取朝南哑变量 - 只要包含"南"字就算朝南
south_mask = orientation_str.str.contains('南')
rent_price_df.loc[valid_mask & south_mask, 'south_dummy'] = 1
rent_price_df.loc[valid_mask & ~south_mask, 'south_dummy'] = 0

# 提取南北通透哑变量 - 同时包含"南"和"北"字
north_south_mask = orientation_str.str.contains('南') & orientation_str.str.contains('北')
rent_price_df.loc[valid_mask & north_south_mask, 'north_south_dummy'] = 1
rent_price_df.loc[valid_mask & ~north_south_mask, 'north_south_dummy'] = 0

print(f"\n已创建的朝向哑变量:")
print(f"south_dummy: 1表示朝南，0表示不朝南，NaN表示缺失")
print(f"north_south_dummy: 1表示南北通透，0表示非南北通透，NaN表示缺失")

print("\n朝向哑变量分布:")
print(f"朝南 (south_dummy=1): {(rent_price_df['south_dummy'] == 1).sum()}")
print(f"不朝南 (south_dummy=0): {(rent_price_df['south_dummy'] == 0).sum()}")
print(f"朝南缺失值: {rent_price_df['south_dummy'].isna().sum()}")

print(f"\n南北通透 (north_south_dummy=1): {(rent_price_df['north_south_dummy'] == 1).sum()}")
print(f"非南北通透 (north_south_dummy=0): {(rent_price_df['north_south_dummy'] == 0).sum()}")
print(f"南北通透缺失值: {rent_price_df['north_south_dummy'].isna().sum()}")

print("朝向哑变量创建完成（保留缺失值）")

朝向分布:
房屋朝向
南              13239
南 北             9887
东南              3305
东               1250
西南              1212
               ...  
东 南 北 东南 东北        1
南 东南 北             1
东南 东 西北 西          1
东 南 东北             1
南 北 东南 西南          1
Name: count, Length: 119, dtype: int64

朝向缺失值数量: 0

已创建的朝向哑变量:
south_dummy: 1表示朝南，0表示不朝南，NaN表示缺失
north_south_dummy: 1表示南北通透，0表示非南北通透，NaN表示缺失

朝向哑变量分布:
朝南 (south_dummy=1): 29313
不朝南 (south_dummy=0): 4704
朝南缺失值: 0

南北通透 (north_south_dummy=1): 10682
非南北通透 (north_south_dummy=0): 23335
南北通透缺失值: 0
朝向哑变量创建完成（保留缺失值）


In [8]:
# 2.7 交易时间变量
# 2.7.1 异常值检查以及处理
null_index = rent_price_df[rent_price_df['交易时间'].isnull()].index
print("空缺行的索引:", null_index.tolist())

# 2.7.2 具体内容
print("交易时间分布:")
print(rent_price_df['交易时间'].value_counts().head(20))

# 2.7.3 提取年份和季度
# 首先确保交易时间是日期格式
rent_price_df['交易时间'] = pd.to_datetime(rent_price_df['交易时间'], errors='coerce')

# 检查转换后的日期范围
print(f"\n交易时间转换后的日期范围:")
print(f"最早日期: {rent_price_df['交易时间'].min()}")
print(f"最晚日期: {rent_price_df['交易时间'].max()}")
print(f"交易时间缺失值数量: {rent_price_df['交易时间'].isna().sum()}")

# 提取年份和季度
rent_price_df['交易年份'] = rent_price_df['交易时间'].dt.year
rent_price_df['交易季度'] = rent_price_df['交易时间'].dt.quarter

# 检查年份和季度分布
print(f"\n交易年份分布:")
print(rent_price_df['交易年份'].value_counts().sort_index())
print(f"\n交易季度分布:")
print(rent_price_df['交易季度'].value_counts().sort_index())

# 删除已存在的交易时间哑变量列
transaction_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('trans_')]
rent_price_df = rent_price_df.drop(columns=transaction_columns_to_drop)

# 创建2024年和2025年的年份哑变量
rent_price_df['trans_2024'] = (rent_price_df['交易年份'] == 2024).astype(int)
rent_price_df['trans_2025'] = (rent_price_df['交易年份'] == 2025).astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['交易时间'].isna().any():
    na_mask = rent_price_df['交易时间'].isna()
    # 为所有交易时间哑变量设置缺失值
    trans_columns = ['trans_2024', 'trans_2025']
    for col in trans_columns:
        rent_price_df.loc[na_mask, col] = np.nan

# 删除中间列
rent_price_df = rent_price_df.drop(['交易年份', '交易季度'], axis=1)

print("\n已创建的交易时间哑变量列:")
transaction_dummy_columns = [col for col in rent_price_df.columns if col.startswith('trans_')]
print(transaction_dummy_columns)

print("\n交易时间哑变量分布:")
for col in transaction_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("交易时间哑变量创建完成")

空缺行的索引: []
交易时间分布:
交易时间
2025-01-19    532
2025-01-17    532
2025-01-20    516
2025-01-18    505
2025-01-23    497
2025-01-16    479
2025-01-24    464
2025-01-26    434
2025-01-25    430
2025-01-21    427
2025-01-27    426
2025-01-31    418
2025-01-28    417
2025-01-30    416
2025-01-22    389
2025-02-01    388
2025-02-03    369
2025-01-29    366
2025-02-02    360
2025-02-04    304
Name: count, dtype: int64

交易时间转换后的日期范围:
最早日期: 2025-01-16 00:00:00
最晚日期: 2025-11-15 00:00:00
交易时间缺失值数量: 0

交易年份分布:
交易年份
2025    34017
Name: count, dtype: int64

交易季度分布:
交易季度
1    15564
2     8497
3     8931
4     1025
Name: count, dtype: int64

已创建的交易时间哑变量列:
['trans_2024', 'trans_2025']

交易时间哑变量分布:
trans_2024: 0个为1, 34017个为0, 0个缺失
trans_2025: 34017个为1, 0个为0, 0个缺失
交易时间哑变量创建完成


In [9]:
# 2.8 建筑结构
# 2.8.1 分布检查
print("建筑结构分布:")
print(rent_price_df['建筑结构'].value_counts())
# 检查缺失值情况
print(f"\n建筑结构缺失值数量: {rent_price_df['建筑结构'].isna().sum()}")

# 2.8.2 创建建筑结构哑变量
# 删除已存在的建筑结构哑变量列
structure_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('structure_')]
rent_price_df = rent_price_df.drop(columns=structure_columns_to_drop)

# 创建建筑结构到英文的映射字典
structure_mapping = {
    '钢混结构': 'steel_concrete',
    '混合结构': 'mixed', 
    '未知结构': 'unknown',
    '砖混结构': 'brick_concrete',
    '框架结构': 'frame',
    '钢结构': 'steel',
    '砖木结构': 'brick_wood'
}

# 将建筑结构转换为英文并创建哑变量
rent_price_df['structure_type_en'] = rent_price_df['建筑结构'].map(structure_mapping)
structure_dummies = pd.get_dummies(rent_price_df['structure_type_en'], prefix='structure')

# 将布尔值转换为整数 (True/False -> 1/0)
structure_dummies = structure_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['建筑结构'].isna().any():
    na_mask = rent_price_df['建筑结构'].isna()
    for col in structure_dummies.columns:
        structure_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框并删除中间列
rent_price_df = pd.concat([rent_price_df, structure_dummies], axis=1)
rent_price_df = rent_price_df.drop('structure_type_en', axis=1)

print(f"\n已创建的建筑结构哑变量:")
structure_dummy_columns = [col for col in rent_price_df.columns if col.startswith('structure_')]
print(structure_dummy_columns)

print("\n建筑结构哑变量分布:")
for col in structure_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("建筑结构哑变量创建完成（保留缺失值）")

建筑结构分布:
建筑结构
钢混结构    25065
混合结构     3460
砖混结构     2715
未知结构     1846
框架结构      651
钢结构       230
砖木结构       36
Name: count, dtype: int64

建筑结构缺失值数量: 14

已创建的建筑结构哑变量:
['structure_brick_concrete', 'structure_brick_wood', 'structure_frame', 'structure_mixed', 'structure_steel', 'structure_steel_concrete', 'structure_unknown']

建筑结构哑变量分布:
structure_brick_concrete: 2715个为1, 31288个为0, 14个缺失
structure_brick_wood: 36个为1, 33967个为0, 14个缺失
structure_frame: 651个为1, 33352个为0, 14个缺失
structure_mixed: 3460个为1, 30543个为0, 14个缺失
structure_steel: 230个为1, 33773个为0, 14个缺失
structure_steel_concrete: 25065个为1, 8938个为0, 14个缺失
structure_unknown: 1846个为1, 32157个为0, 14个缺失
建筑结构哑变量创建完成（保留缺失值）


In [10]:
# 2.9 梯户比例
# 2.9.1 分布检查
print("梯户比例分布:")
print(rent_price_df['梯户比例'].value_counts())
# 检查缺失值情况
print(f"\n梯户比例缺失值数量: {rent_price_df['梯户比例'].isna().sum()}")

# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['梯户比例'].value_counts())
# pd.reset_option('display.max_rows')

# 2.9.2 拆分梯户比例为梯数和户数
# 删除已存在的梯户相关列
ladder_house_columns_to_drop = [col for col in rent_price_df.columns if col in ['梯数', '户数']]
rent_price_df = rent_price_df.drop(columns=ladder_house_columns_to_drop)

def split_ladder_household(ratio_str):
    """
    将梯户比例字符串拆分为梯数和户数
    例如："一梯两户" -> (1, 2), "两梯四户" -> (2, 4)
    """
    # 检查是否为NaN或非字符串类型
    if pd.isna(ratio_str) or not isinstance(ratio_str, str):
        return None, None
    
    # 中文数字到阿拉伯数字的映射
    chinese_to_arabic = {
        '一': 1, '两': 2, '二': 2, '三': 3, '四': 4, '五': 5,
        '六': 6, '七': 7, '八': 8, '九': 9, '十': 10,
        '十一': 11, '十二': 12, '十三': 13, '十四': 14, '十五': 15,
        '十六': 16, '十七': 17, '十八': 18, '十九': 19, '二十': 20,
        '二十一': 21, '二十二': 22, '二十三': 23, '二十四': 24, '二十五': 25,
        '二十六': 26, '二十七': 27, '二十八': 28, '二十九': 29, '三十': 30,
        '三十一': 31, '三十二': 32, '三十三': 33, '三十四': 34, '三十五': 35,
        '三十六': 36, '三十七': 37, '三十八': 38, '三十九': 39, '四十': 40,
        '四十一': 41, '四十二': 42, '四十三': 43, '四十四': 44, '四十五': 45,
        '四十六': 46, '四十七': 47, '四十八': 48, '四十九': 49, '五十': 50,
        '五十一': 51, '五十二': 52, '五十三': 53, '五十四': 54, '五十五': 55,
        '五十六': 56, '五十七': 57, '五十八': 58, '五十九': 59, '六十': 60,
        '六十一': 61, '六十二': 62, '六十三': 63, '六十四': 64, '六十五': 65,
        '六十六': 66, '六十七': 67, '六十八': 68, '六十九': 69, '七十': 70,
        '七十一': 71, '七十二': 72, '七十三': 73, '七十四': 74, '七十五': 75,
        '七十六': 76, '七十七': 77, '七十八': 78, '七十九': 79, '八十': 80,
        '八十一': 81, '八十二': 82, '八十三': 83, '八十四': 84, '八十五': 85,
        '八十六': 86, '八十七': 87, '八十八': 88, '八十九': 89, '九十': 90,
        '九十一': 91, '九十二': 92, '九十三': 93, '九十四': 94, '九十五': 95,
        '九十六': 96, '九十七': 97, '九十八': 98, '九十九': 99, '一百': 100,
        '一百零一': 101, '一百零二': 102, '一百零三': 103, '一百零四': 104, '一百零五': 105,
        '一百零六': 106, '一百一十二': 112, '一百九十一': 191
    }
    
    try:
        # 查找"梯"和"户"的位置
        ladder_idx = ratio_str.find('梯')
        household_idx = ratio_str.find('户')
        
        if ladder_idx == -1 or household_idx == -1:
            return None, None
            
        # 提取梯数和户数的中文表示
        ladder_chinese = ratio_str[:ladder_idx]
        household_chinese = ratio_str[ladder_idx+1:household_idx]
        
        # 转换为阿拉伯数字
        ladder_num = chinese_to_arabic.get(ladder_chinese)
        household_num = chinese_to_arabic.get(household_chinese)
        
        return ladder_num, household_num
        
    except Exception as e:
        print(f"处理梯户比例 '{ratio_str}' 时出错: {e}")
        return None, None

# 应用函数拆分梯户比例
rent_price_df['梯数'] = rent_price_df['梯户比例'].apply(lambda x: split_ladder_household(x)[0])
rent_price_df['户数'] = rent_price_df['梯户比例'].apply(lambda x: split_ladder_household(x)[1])

# 检查拆分结果
print("\n梯数和户数拆分结果:")
print(f"梯数唯一值: {sorted(rent_price_df['梯数'].dropna().unique())}")
print(f"户数唯一值: {sorted(rent_price_df['户数'].dropna().unique())}")

# 输出梯数和户数的分布情况
print("\n梯数分布:")
ladder_valid_count = rent_price_df['梯数'].notna().sum()
ladder_na_count = rent_price_df['梯数'].isna().sum()
print(f"梯数: {ladder_valid_count}个有效值, {ladder_na_count}个缺失值")

print("\n户数分布:")
household_valid_count = rent_price_df['户数'].notna().sum()
household_na_count = rent_price_df['户数'].isna().sum()
print(f"户数: {household_valid_count}个有效值, {household_na_count}个缺失值")

梯户比例分布:
梯户比例
一梯两户      8356
两梯四户      3918
一梯四户      3098
一梯三户      2871
两梯六户      1993
          ... 
两梯二十七户       1
一梯五十九户       1
六梯八户         1
五梯十七户        1
五梯二十二户       1
Name: count, Length: 256, dtype: int64

梯户比例缺失值数量: 635

梯数和户数拆分结果:
梯数唯一值: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(12.0), np.float64(13.0), np.float64(15.0), np.float64(16.0), np.float64(18.0), np.float64(19.0), np.float64(20.0)]
户数唯一值: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0), np.float64(25.0), n

In [11]:
# 2.10 电梯
# 2.10.1 分布检查
print("电梯分布:")
print(rent_price_df['配备电梯'].value_counts())

# 2.10.2 异常值处理
# 检查是否有异常值或缺失值
print(f"\n电梯缺失值数量: {rent_price_df['配备电梯'].isna().sum()}")

# 2.10.3 创建电梯哑变量
# 删除已存在的电梯哑变量列
elevator_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('elevator_')]
rent_price_df = rent_price_df.drop(columns=elevator_columns_to_drop)

# 创建电梯哑变量
# 以"无电梯"为基准，创建"有电梯"的哑变量
# 保留缺失值为NaN，不进行填充
rent_price_df['elevator_yes'] = rent_price_df['配备电梯'].map({'无': 0, '有': 1})

print(f"\n已创建的电梯哑变量:")
print(f"elevator_yes: 1表示有电梯，0表示无电梯，NaN表示缺失")

print("\n电梯哑变量分布:")
print(f"无电梯 (elevator_yes=0): {(rent_price_df['elevator_yes'] == 0).sum()}")
print(f"有电梯 (elevator_yes=1): {(rent_price_df['elevator_yes'] == 1).sum()}")
print(f"缺失值: {rent_price_df['elevator_yes'].isna().sum()}")

print("电梯哑变量创建完成（保留缺失值）")

电梯分布:
配备电梯
有    21713
无     8212
Name: count, dtype: int64

电梯缺失值数量: 4092

已创建的电梯哑变量:
elevator_yes: 1表示有电梯，0表示无电梯，NaN表示缺失

电梯哑变量分布:
无电梯 (elevator_yes=0): 8212
有电梯 (elevator_yes=1): 21713
缺失值: 4092
电梯哑变量创建完成（保留缺失值）


In [12]:
# 2.11 交易权属
# 2.11.1 分布检查
print("交易权属分布:")
print(rent_price_df['交易权属'].value_counts())
# 检查缺失值情况
print(f"\n交易权属缺失值数量: {rent_price_df['交易权属'].isna().sum()}")

# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['交易权属'].value_counts())
# pd.reset_option('display.max_rows')

# 2.11.2 创建交易权属哑变量（商品房 vs 非商品房）
# 删除已存在的交易权属哑变量列
transaction_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('transaction_')]
rent_price_df = rent_price_df.drop(columns=transaction_columns_to_drop)

# 定义商品房和非商品房的分类
commercial_properties = ['商品房']
non_commercial_properties = [
    '已购公房', '动迁安置房', '二类经济适用房', '房改房', '集资房', 
    '一类经济适用房', '拆迁还建房', '经济适用房', '私产', '央产房', 
    '限价商品房', '定向安置房', '售后公房', '回迁房'
]

# 创建交易权属类型映射
def map_transaction_type(transaction):
    if pd.isna(transaction):
        return None
    elif transaction in commercial_properties:
        return 'commercial'
    elif transaction in non_commercial_properties:
        return 'non_commercial'
    else:
        return 'other'

# 将交易权属转换为类型并创建哑变量
rent_price_df['transaction_type_en'] = rent_price_df['交易权属'].apply(map_transaction_type)
transaction_dummies = pd.get_dummies(rent_price_df['transaction_type_en'], prefix='transaction')

# 将布尔值转换为整数 (True/False -> 1/0)
transaction_dummies = transaction_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['交易权属'].isna().any():
    na_mask = rent_price_df['交易权属'].isna()
    for col in transaction_dummies.columns:
        transaction_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框并删除中间列
rent_price_df = pd.concat([rent_price_df, transaction_dummies], axis=1)
rent_price_df = rent_price_df.drop('transaction_type_en', axis=1)

print(f"\n已创建的交易权属哑变量:")
transaction_dummy_columns = [col for col in rent_price_df.columns if col.startswith('transaction_')]
print(transaction_dummy_columns)

print("\n交易权属哑变量分布:")
for col in transaction_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

# 验证分类是否正确
print(f"\n分类验证:")
print(f"商品房数量: {len(rent_price_df[rent_price_df['交易权属'].isin(commercial_properties)])}")
print(f"非商品房数量: {len(rent_price_df[rent_price_df['交易权属'].isin(non_commercial_properties)])}")
print(f"总计: {len(rent_price_df[rent_price_df['交易权属'].isin(commercial_properties + non_commercial_properties)])}")

print("交易权属哑变量创建完成（保留缺失值）")

交易权属分布:
交易权属
商品房        31161
已购公房        1007
动迁安置房        612
一类经济适用房      225
二类经济适用房      215
房改房          168
集资房          152
央产房          143
限价商品房         93
拆迁还建房         62
经济适用房         54
定向安置房         52
私产            40
售后公房          26
自住型商品房         7
Name: count, dtype: int64

交易权属缺失值数量: 0

已创建的交易权属哑变量:
['transaction_commercial', 'transaction_non_commercial', 'transaction_other']

交易权属哑变量分布:
transaction_commercial: 31161个为1, 2856个为0, 0个缺失
transaction_non_commercial: 2849个为1, 31168个为0, 0个缺失
transaction_other: 7个为1, 34010个为0, 0个缺失

分类验证:
商品房数量: 31161
非商品房数量: 2849
总计: 34010
交易权属哑变量创建完成（保留缺失值）


In [13]:
# 2.12 房屋用途
# 2.11.1 分布检查
print("房屋用途分布:")
print(rent_price_df['房屋用途'].value_counts())
# 检查缺失值情况
print(f"\n房屋用途缺失值数量: {rent_price_df['房屋用途'].isna().sum()}")

# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['房屋用途'].value_counts())
# pd.reset_option('display.max_rows')

# 2.12.2 创建房屋用途哑变量
# 删除已存在的房屋用途哑变量列
usage_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('usage_')]
rent_price_df = rent_price_df.drop(columns=usage_columns_to_drop)

# 创建房屋用途到分类的映射字典
usage_mapping = {
    # 高端住宅
    '别墅': 'high_end_residential',
    '花园洋房': 'high_end_residential',
    '四合院': 'high_end_residential',
    '新式里弄': 'high_end_residential',
    
    # 普通住宅
    '普通住宅': 'ordinary_residential',
    '公寓': 'ordinary_residential',
    '公寓/住宅': 'ordinary_residential',
    '公寓（住宅）': 'ordinary_residential',
    '住宅式公寓': 'ordinary_residential',
    '老公寓': 'ordinary_residential',
    
    # 商住混合
    '商住两用': 'commercial_residential',
    '商务公寓': 'commercial_residential',
    '酒店式公寓': 'commercial_residential',
    '商务型公寓': 'commercial_residential',
    '公寓/公寓': 'commercial_residential',
    
    # 商业办公
    '商业办公类': 'commercial_office',
    '写字楼': 'commercial_office',
    '商业': 'commercial_office',
    '底商': 'commercial_office',
    
    # 其他特殊用途
    '车库': 'other_special'
}

# 将房屋用途转换为分类并创建哑变量
rent_price_df['usage_type_en'] = rent_price_df['房屋用途'].map(usage_mapping)
usage_dummies = pd.get_dummies(rent_price_df['usage_type_en'], prefix='usage')

# 将布尔值转换为整数 (True/False -> 1/0)
usage_dummies = usage_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['房屋用途'].isna().any():
    na_mask = rent_price_df['房屋用途'].isna()
    for col in usage_dummies.columns:
        usage_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框并删除中间列
rent_price_df = pd.concat([rent_price_df, usage_dummies], axis=1)
rent_price_df = rent_price_df.drop('usage_type_en', axis=1)

print(f"\n已创建的房屋用途哑变量:")
usage_dummy_columns = [col for col in rent_price_df.columns if col.startswith('usage_')]
print(usage_dummy_columns)

print("\n房屋用途哑变量分布:")
for col in usage_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("房屋用途哑变量创建完成（保留缺失值）")

房屋用途分布:
房屋用途
普通住宅      32979
商住两用        376
别墅          259
商业办公类       250
公寓           78
商务公寓         29
酒店式公寓        16
车库           14
新式里弄          6
公寓/公寓         3
花园洋房          2
公寓（住宅）        1
公寓/住宅         1
商务型公寓         1
老公寓           1
商业            1
Name: count, dtype: int64

房屋用途缺失值数量: 0

已创建的房屋用途哑变量:
['usage_commercial_office', 'usage_commercial_residential', 'usage_high_end_residential', 'usage_ordinary_residential', 'usage_other_special']

房屋用途哑变量分布:
usage_commercial_office: 251个为1, 33766个为0, 0个缺失
usage_commercial_residential: 425个为1, 33592个为0, 0个缺失
usage_high_end_residential: 267个为1, 33750个为0, 0个缺失
usage_ordinary_residential: 33060个为1, 957个为0, 0个缺失
usage_other_special: 14个为1, 34003个为0, 0个缺失
房屋用途哑变量创建完成（保留缺失值）


In [14]:
# 2.13 房屋年限
# 2.13.1 分布检查
print("房屋年限分布:")
print(rent_price_df['房屋年限'].value_counts())
# 检查缺失值情况
print(f"\n房屋年限缺失值数量: {rent_price_df['房屋年限'].isna().sum()}")

# 2.13.2 创建房屋年限哑变量
# 删除已存在的房屋年限哑变量列
house_age_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('house_age_')]
rent_price_df = rent_price_df.drop(columns=house_age_columns_to_drop)

# 创建房屋年限到英文的映射字典
house_age_mapping = {
    '满五年': 'over_5_years',
    '满两年': 'over_2_years', 
    '未满两年': 'under_2_years'
}

# 将房屋年限转换为英文并创建哑变量
rent_price_df['house_age_type_en'] = rent_price_df['房屋年限'].map(house_age_mapping)
house_age_dummies = pd.get_dummies(rent_price_df['house_age_type_en'], prefix='house_age')

# 将布尔值转换为整数 (True/False -> 1/0)
house_age_dummies = house_age_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['房屋年限'].isna().any():
    na_mask = rent_price_df['房屋年限'].isna()
    for col in house_age_dummies.columns:
        house_age_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框并删除中间列
rent_price_df = pd.concat([rent_price_df, house_age_dummies], axis=1)
rent_price_df = rent_price_df.drop('house_age_type_en', axis=1)

print(f"\n已创建的房屋年限哑变量:")
house_age_dummy_columns = [col for col in rent_price_df.columns if col.startswith('house_age_')]
print(house_age_dummy_columns)

print("\n房屋年限哑变量分布:")
for col in house_age_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("房屋年限哑变量创建完成（保留缺失值）")

房屋年限分布:
房屋年限
满五年     15065
满两年      5241
未满两年     2415
Name: count, dtype: int64

房屋年限缺失值数量: 11296

已创建的房屋年限哑变量:
['house_age_over_2_years', 'house_age_over_5_years', 'house_age_under_2_years']

房屋年限哑变量分布:
house_age_over_2_years: 5241个为1, 17480个为0, 11296个缺失
house_age_over_5_years: 15065个为1, 7656个为0, 11296个缺失
house_age_under_2_years: 2415个为1, 20306个为0, 11296个缺失
房屋年限哑变量创建完成（保留缺失值）


In [15]:
# 2.14 地铁
# 2.14.1 分布检查
print("房屋优势分布:")
print(rent_price_df['房屋优势'].value_counts())
# 检查缺失值情况
print(f"\n房屋优势缺失值数量: {rent_price_df['房屋优势'].isna().sum()}")

# 2.14.2 创建地铁哑变量
# 删除已存在的地铁变量列
if 'subway' in rent_price_df.columns:
    rent_price_df = rent_price_df.drop('subway', axis=1)

# 创建地铁哑变量
def has_subway(advantage):
    """
    检查房屋优势中是否包含"地铁"
    """
    if pd.isna(advantage):
        return np.nan
    elif '地铁' in str(advantage):
        return 1
    else:
        return 0

# 应用函数创建地铁哑变量
rent_price_df['subway'] = rent_price_df['房屋优势'].apply(has_subway)

# 检查地铁哑变量分布
print(f"\n地铁哑变量分布:")
subway_valid_count = rent_price_df['subway'].notna().sum()
subway_true_count = (rent_price_df['subway'] == 1).sum()
subway_false_count = (rent_price_df['subway'] == 0).sum()
subway_na_count = rent_price_df['subway'].isna().sum()

print(f"subway: {subway_true_count}个为1, {subway_false_count}个为0, {subway_na_count}个缺失")

print("地铁哑变量创建完成（保留缺失值）")

房屋优势分布:
房屋优势
地铁、房本满五年        2307
装修、房本满五年        2201
、房本满五年          2140
、               1876
地铁、装修、房本满五年     1874
装修、房本满五年、       1707
装修、             1587
、房本满五年、         1345
地铁、装修、房本满五年、    1317
装修              1306
地铁、房本满五年、       1164
地铁、             1075
装修、房本满两年、        910
房本满五年            846
、房本满两年、          711
装修、房本满两年         681
、房本满两年           674
地铁               614
地铁、房本满两年         593
地铁、装修、           525
房本满两年            436
地铁、装修            386
地铁、装修、房本满两年      368
地铁、装修、房本满两年、     360
地铁、房本满两年、        314
房本满五年、           167
房本满两年、           154
Name: count, dtype: int64

房屋优势缺失值数量: 6379

地铁哑变量分布:
subway: 10897个为1, 16741个为0, 6379个缺失
地铁哑变量创建完成（保留缺失值）


In [16]:
# 2.15 建筑年代
# 2.15.1 分布检查
print("建筑年代分布:")
print(rent_price_df['建筑年代'].value_counts()) 
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['建筑年代'].value_counts())
# pd.reset_option('display.max_rows')

# 2.15.2 缺失值检查
print(f"\n建筑年代缺失值数量: {rent_price_df['建筑年代'].isna().sum()}")

# 2.15.3 创建房龄变量
# 删除已存在的房龄变量列
age_columns_to_drop = [col for col in rent_price_df.columns if col in ['building_age']]
rent_price_df = rent_price_df.drop(columns=age_columns_to_drop)

# 初始化房龄变量为NaN
rent_price_df['building_age'] = np.nan

def extract_building_age(building_year_str):
    """
    从建筑年代字符串中提取建筑结束年份并计算房龄
    处理逻辑:
    - 单个年份: "2008年" → 结束年份=2008
    - 精确区间: "2008-2014年" → 结束年份=2014
    - 跨度区间: "1980-1999年" → 结束年份=1999
    - 房龄 = 2025 - 结束年份 (假设数据收集年份为2025年)
    """
    if pd.isna(building_year_str):
        return np.nan
    
    building_year_str = str(building_year_str).strip()
    
    try:
        # 假设数据收集年份为2025年
        current_year = 2025
        
        # 处理单个年份格式
        if '年' in building_year_str and '-' not in building_year_str:
            year = int(building_year_str.replace('年', ''))
            return current_year - year
        
        # 处理区间格式
        elif '-' in building_year_str and '年' in building_year_str:
            # 提取年份范围
            year_range = building_year_str.replace('年', '').split('-')
            
            # 确保有两个年份
            if len(year_range) == 2:
                start_year = int(year_range[0])
                end_year = int(year_range[1])
                
                # 使用结束年份计算房龄
                return current_year - end_year
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析建筑年代 '{building_year_str}': {e}")
        return np.nan

# 只对非空建筑年代进行处理
valid_mask = rent_price_df['建筑年代'].notna()

# 应用转换函数
for idx in rent_price_df[valid_mask].index:
    building_year_str = rent_price_df.loc[idx, '建筑年代']
    building_age = extract_building_age(building_year_str)
    rent_price_df.loc[idx, 'building_age'] = building_age

print(f"\n已创建的房龄变量:")
print(f"building_age: 房龄(年)，基于建筑结束年份计算，NaN表示缺失")

print("\n房龄变量统计:")
print(f"非缺失值数量: {rent_price_df['building_age'].notna().sum()}")
if rent_price_df['building_age'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['building_age'].mean():.2f}")
    print(f"标准差: {rent_price_df['building_age'].std():.2f}")
    print(f"最小值: {rent_price_df['building_age'].min()}")
    print(f"最大值: {rent_price_df['building_age'].max()}")
    print(f"分布:")
    print(rent_price_df['building_age'].value_counts().sort_index().head(20))  # 只显示前20个最常见的房龄
print(f"缺失值数量: {rent_price_df['building_age'].isna().sum()}")

print("房龄变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n建筑年代转换示例:")
sample_data = rent_price_df[valid_mask][['建筑年代', 'building_age']].head(10)
print(sample_data)

建筑年代分布:
建筑年代
2000-2012年    343
2015-2018年    303
2000-2018年    292
2018-2019年    290
2006年         288
             ... 
1974-1979年      1
2018-2021年      1
1970-1979年      1
1985-2001年      1
1998-2012年      1
Name: count, Length: 716, dtype: int64

建筑年代缺失值数量: 9406

已创建的房龄变量:
building_age: 房龄(年)，基于建筑结束年份计算，NaN表示缺失

房龄变量统计:
非缺失值数量: 24611
平均值: 15.99
标准差: 8.52
最小值: 3.0
最大值: 89.0
分布:
building_age
3.0      269
4.0      101
5.0      857
6.0     1258
7.0     1893
8.0     1422
9.0     1564
10.0    1223
11.0    1051
12.0     928
13.0    1050
14.0     619
15.0     898
16.0     757
17.0    1280
18.0     506
19.0     871
20.0     714
21.0     805
22.0     813
Name: count, dtype: int64
缺失值数量: 9406
房龄变量创建完成（保留缺失值）

建筑年代转换示例:
         建筑年代  building_age
0  2002-2006年          19.0
1  2007-2009年          16.0
2  1999-2001年          24.0
3  2004-2008年          17.0
4  1993-1997年          28.0
5  2003-2006年          19.0
6  2014-2019年           6.0
7  1993-2009年          16.0
8  2018-2022年           3.

In [17]:
# 2.16 小区规模（房屋总数和栋数）
# 2.16.1 分布检查
print("房屋总数分布:")
print(rent_price_df['房屋总数'].value_counts().head(20))  # 只显示前20个最常见的值

print("\n楼栋总数分布:")
print(rent_price_df['楼栋总数'].value_counts().head(20))  # 只显示前20个最常见的值

# 2.16.2 缺失值检查
print(f"\n房屋总数缺失值数量: {rent_price_df['房屋总数'].isna().sum()}")
print(f"楼栋总数缺失值数量: {rent_price_df['楼栋总数'].isna().sum()}")

# 2.16.3 创建小区规模变量
# 删除已存在的小区规模变量列
community_columns_to_drop = [col for col in rent_price_df.columns if 
                            col in ['household_total', 'building_total']]
rent_price_df = rent_price_df.drop(columns=community_columns_to_drop)

# 初始化小区规模变量为NaN
rent_price_df['household_total'] = np.nan
rent_price_df['building_total'] = np.nan

def extract_community_scale(household_str, building_str):
    """
    从房屋总数和楼栋总数字符串中提取数值
    处理逻辑:
    - 提取数字部分，去除"户"、"栋"等单位
    - 转换为数值类型
    """
    household_value = np.nan
    building_value = np.nan
    
    # 处理房屋总数
    if pd.notna(household_str):
        household_str = str(household_str).strip()
        try:
            # 提取数字部分
            household_match = re.search(r'(\d+)', household_str)
            if household_match:
                household_value = int(household_match.group(1))
        except (ValueError, AttributeError) as e:
            print(f"警告: 无法解析房屋总数 '{household_str}': {e}")
    
    # 处理楼栋总数
    if pd.notna(building_str):
        building_str = str(building_str).strip()
        try:
            # 提取数字部分
            building_match = re.search(r'(\d+)', building_str)
            if building_match:
                building_value = int(building_match.group(1))
        except (ValueError, AttributeError) as e:
            print(f"警告: 无法解析楼栋总数 '{building_str}': {e}")
    
    return household_value, building_value

# 只对非空数据进行处理
valid_household_mask = rent_price_df['房屋总数'].notna()
valid_building_mask = rent_price_df['楼栋总数'].notna()

# 应用转换函数
for idx in rent_price_df.index:
    household_str = rent_price_df.loc[idx, '房屋总数']
    building_str = rent_price_df.loc[idx, '楼栋总数']
    
    household_value, building_value = extract_community_scale(household_str, building_str)
    
    if pd.notna(household_value):
        rent_price_df.loc[idx, 'household_total'] = household_value
    
    if pd.notna(building_value):
        rent_price_df.loc[idx, 'building_total'] = building_value

print(f"\n已创建的小区规模变量:")
print(f"household_total: 房屋总户数，NaN表示缺失")
print(f"building_total: 楼栋总数，NaN表示缺失")

print("\n小区规模变量统计:")
print("household_total:")
print(f"  非缺失值数量: {rent_price_df['household_total'].notna().sum()}")
if rent_price_df['household_total'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['household_total'].mean():.2f}")
    print(f"  标准差: {rent_price_df['household_total'].std():.2f}")
    print(f"  最小值: {rent_price_df['household_total'].min()}")
    print(f"  最大值: {rent_price_df['household_total'].max()}")
print(f"  缺失值数量: {rent_price_df['household_total'].isna().sum()}")

print("\nbuilding_total:")
print(f"  非缺失值数量: {rent_price_df['building_total'].notna().sum()}")
if rent_price_df['building_total'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['building_total'].mean():.2f}")
    print(f"  标准差: {rent_price_df['building_total'].std():.2f}")
    print(f"  最小值: {rent_price_df['building_total'].min()}")
    print(f"  最大值: {rent_price_df['building_total'].max()}")
print(f"  缺失值数量: {rent_price_df['building_total'].isna().sum()}")

print("小区规模变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n小区规模转换示例:")
sample_data = rent_price_df[['房屋总数', '楼栋总数', 'household_total', 'building_total']].head(10)
print(sample_data)

房屋总数分布:
房屋总数
10505户    176
7666户     170
5184户     144
9342户     142
6458户     139
7435户     138
6070户     135
3139户     123
6617户     118
4644户     114
11953户    112
7387户     107
5767户     104
15户       102
6830户     100
4193户      96
2283户      95
1842户      95
3946户      94
2028户      91
Name: count, dtype: int64

楼栋总数分布:
楼栋总数
10栋    1169
7栋     1151
5栋     1093
6栋     1068
8栋     1013
2栋      993
4栋      984
3栋      964
13栋     931
12栋     889
9栋      888
16栋     834
11栋     770
23栋     753
19栋     722
1栋      715
15栋     708
18栋     651
17栋     640
14栋     627
Name: count, dtype: int64

房屋总数缺失值数量: 3715
楼栋总数缺失值数量: 3715

已创建的小区规模变量:
household_total: 房屋总户数，NaN表示缺失
building_total: 楼栋总数，NaN表示缺失

小区规模变量统计:
household_total:
  非缺失值数量: 30302
  平均值: 1962.30
  标准差: 1924.48
  最小值: 1.0
  最大值: 12669.0
  缺失值数量: 3715

building_total:
  非缺失值数量: 30302
  平均值: 37.57
  标准差: 72.09
  最小值: 1.0
  最大值: 734.0
  缺失值数量: 3715
小区规模变量创建完成（保留缺失值）

小区规模转换示例:
    房屋总数 楼栋总数  household_total  building_total
0   458户

In [18]:
# 2.16 建筑年代
# 2.16.1 分布检查
print("建筑年代分布:")
print(rent_price_df['建筑年代'].value_counts()) 
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['建筑年代'].value_counts())
# pd.reset_option('display.max_rows')

# 2.16.2 缺失值检查
print(f"\n建筑年代缺失值数量: {rent_price_df['建筑年代'].isna().sum()}")

# 2.16.3 创建房龄变量
# 删除已存在的房龄变量列
age_columns_to_drop = [col for col in rent_price_df.columns if col in ['building_age']]
rent_price_df = rent_price_df.drop(columns=age_columns_to_drop)

# 初始化房龄变量为NaN
rent_price_df['building_age'] = np.nan

def extract_building_age(building_year_str):
    """
    从建筑年代字符串中提取建筑结束年份并计算房龄
    处理逻辑:
    - 单个年份: "2008年" → 结束年份=2008
    - 精确区间: "2008-2014年" → 结束年份=2014
    - 跨度区间: "1980-1999年" → 结束年份=1999
    - 房龄 = 2025 - 结束年份 (假设数据收集年份为2025年)
    """
    if pd.isna(building_year_str):
        return np.nan
    
    building_year_str = str(building_year_str).strip()
    
    try:
        # 假设数据收集年份为2025年
        current_year = 2025
        
        # 处理单个年份格式
        if '年' in building_year_str and '-' not in building_year_str:
            year = int(building_year_str.replace('年', ''))
            return current_year - year
        
        # 处理区间格式
        elif '-' in building_year_str and '年' in building_year_str:
            # 提取年份范围
            year_range = building_year_str.replace('年', '').split('-')
            
            # 确保有两个年份
            if len(year_range) == 2:
                start_year = int(year_range[0])
                end_year = int(year_range[1])
                
                # 使用结束年份计算房龄
                return current_year - end_year
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析建筑年代 '{building_year_str}': {e}")
        return np.nan

# 只对非空建筑年代进行处理
valid_mask = rent_price_df['建筑年代'].notna()

# 应用转换函数
for idx in rent_price_df[valid_mask].index:
    building_year_str = rent_price_df.loc[idx, '建筑年代']
    building_age = extract_building_age(building_year_str)
    rent_price_df.loc[idx, 'building_age'] = building_age

print(f"\n已创建的房龄变量:")
print(f"building_age: 房龄(年)，基于建筑结束年份计算，NaN表示缺失")

print("\n房龄变量统计:")
print(f"非缺失值数量: {rent_price_df['building_age'].notna().sum()}")
if rent_price_df['building_age'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['building_age'].mean():.2f}")
    print(f"标准差: {rent_price_df['building_age'].std():.2f}")
    print(f"最小值: {rent_price_df['building_age'].min()}")
    print(f"最大值: {rent_price_df['building_age'].max()}")
    print(f"分布:")
    print(rent_price_df['building_age'].value_counts().sort_index().head(20))  # 只显示前20个最常见的房龄
print(f"缺失值数量: {rent_price_df['building_age'].isna().sum()}")

print("房龄变量创建完成（保留缺失值）")

# 显示一些转换示例
print("\n建筑年代转换示例:")
sample_data = rent_price_df[valid_mask][['建筑年代', 'building_age']].head(10)
print(sample_data)

建筑年代分布:
建筑年代
2000-2012年    343
2015-2018年    303
2000-2018年    292
2018-2019年    290
2006年         288
             ... 
1974-1979年      1
2018-2021年      1
1970-1979年      1
1985-2001年      1
1998-2012年      1
Name: count, Length: 716, dtype: int64

建筑年代缺失值数量: 9406

已创建的房龄变量:
building_age: 房龄(年)，基于建筑结束年份计算，NaN表示缺失

房龄变量统计:
非缺失值数量: 24611
平均值: 15.99
标准差: 8.52
最小值: 3.0
最大值: 89.0
分布:
building_age
3.0      269
4.0      101
5.0      857
6.0     1258
7.0     1893
8.0     1422
9.0     1564
10.0    1223
11.0    1051
12.0     928
13.0    1050
14.0     619
15.0     898
16.0     757
17.0    1280
18.0     506
19.0     871
20.0     714
21.0     805
22.0     813
Name: count, dtype: int64
缺失值数量: 9406
房龄变量创建完成（保留缺失值）

建筑年代转换示例:
         建筑年代  building_age
0  2002-2006年          19.0
1  2007-2009年          16.0
2  1999-2001年          24.0
3  2004-2008年          17.0
4  1993-1997年          28.0
5  2003-2006年          19.0
6  2014-2019年           6.0
7  1993-2009年          16.0
8  2018-2022年           3.

In [19]:
# 2.17 绿化率和容积率
# 2.17.1 分布检查
print("绿化率分布:")
print(rent_price_df['绿化率'].value_counts().head(10))  # 只显示前10个最常见的值

print("\n容积率分布:")
print(rent_price_df['容积率'].value_counts().head(10))  # 只显示前10个最常见的值

# 2.17.2 缺失值检查
print(f"\n绿化率缺失值数量: {rent_price_df['绿化率'].isna().sum()}")
print(f"容积率缺失值数量: {rent_price_df['容积率'].isna().sum()}")

# 2.17.3 创建绿化率和容积率变量
# 删除已存在的绿化率和容积率变量列
greening_columns_to_drop = [col for col in rent_price_df.columns if 
                           col in ['greening_rate', 'plot_ratio']]
rent_price_df = rent_price_df.drop(columns=greening_columns_to_drop)

# 创建绿化率和容积率变量
# 保留缺失值为NaN，不进行填充

# 处理绿化率 - 移除百分号并转换为浮点数
rent_price_df['greening_rate'] = np.nan
valid_greening_mask = rent_price_df['绿化率'].notna()
rent_price_df.loc[valid_greening_mask, 'greening_rate'] = (
    rent_price_df.loc[valid_greening_mask, '绿化率']
    .astype(str)
    .str.replace('%', '')
    .astype(float) / 100
)

# 处理容积率 - 直接转换为浮点数
rent_price_df['plot_ratio'] = np.nan
valid_plot_mask = rent_price_df['容积率'].notna()
rent_price_df.loc[valid_plot_mask, 'plot_ratio'] = (
    rent_price_df.loc[valid_plot_mask, '容积率'].astype(float)
)

print(f"\n已创建的变量:")
print(f"greening_rate: 绿化率（小数形式），NaN表示缺失")
print(f"plot_ratio: 容积率，NaN表示缺失")

print("\n绿化率和容积率变量统计:")
print("greening_rate:")
print(f"  非缺失值数量: {rent_price_df['greening_rate'].notna().sum()}")
if rent_price_df['greening_rate'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['greening_rate'].mean():.3f}")
    print(f"  标准差: {rent_price_df['greening_rate'].std():.3f}")
    print(f"  最小值: {rent_price_df['greening_rate'].min():.3f}")
    print(f"  最大值: {rent_price_df['greening_rate'].max():.3f}")
print(f"  缺失值数量: {rent_price_df['greening_rate'].isna().sum()}")

print("\nplot_ratio:")
print(f"  非缺失值数量: {rent_price_df['plot_ratio'].notna().sum()}")
if rent_price_df['plot_ratio'].notna().sum() > 0:
    print(f"  平均值: {rent_price_df['plot_ratio'].mean():.3f}")
    print(f"  标准差: {rent_price_df['plot_ratio'].std():.3f}")
    print(f"  最小值: {rent_price_df['plot_ratio'].min():.3f}")
    print(f"  最大值: {rent_price_df['plot_ratio'].max():.3f}")
print(f"  缺失值数量: {rent_price_df['plot_ratio'].isna().sum()}")

print("绿化率和容积率变量创建完成（保留缺失值）")

绿化率分布:
绿化率
0.30    6594
0.35    6243
0.40    1973
0.45     961
0.20     893
0.25     748
0.10     585
0.36     472
0.50     437
0.38     344
Name: count, dtype: int64

容积率分布:
容积率
2.0    2555
2.5    2007
3.0    1742
1.5     969
1.8     738
2.2     659
1.0     618
1.6     579
1.2     566
3.2     512
Name: count, dtype: int64

绿化率缺失值数量: 9180
容积率缺失值数量: 9303

已创建的变量:
greening_rate: 绿化率（小数形式），NaN表示缺失
plot_ratio: 容积率，NaN表示缺失

绿化率和容积率变量统计:
greening_rate:
  非缺失值数量: 24837
  平均值: 0.004
  标准差: 0.021
  最小值: 0.000
  最大值: 1.050
  缺失值数量: 9180

plot_ratio:
  非缺失值数量: 24714
  平均值: 2.613
  标准差: 1.553
  最小值: 0.020
  最大值: 35.000
  缺失值数量: 9303
绿化率和容积率变量创建完成（保留缺失值）


In [20]:
# 2.18 物业费处理
# 2.18.1 分布检查
print("物业费分布:")
print(rent_price_df['物业费'].value_counts())
print(f"\n物业费缺失值数量: {rent_price_df['物业费'].isna().sum()}")

# 2.18.2 创建物业费变量
# 删除已存在的物业费变量列
property_columns_to_drop = [col for col in rent_price_df.columns if 
                           col in ['property_fee_avg']]
rent_price_df = rent_price_df.drop(columns=property_columns_to_drop)

# 初始化物业费变量为NaN
rent_price_df['property_fee_avg'] = np.nan

def extract_property_fee(property_fee_str):
    """
    从物业费字符串中提取平均值
    处理逻辑:
    - 单个数值: "2.8元/月/㎡" → 平均值=2.8
    - 区间数值: "2.38-2.98元/月/㎡" → 平均值=(2.38+2.98)/2=2.68
    - 特殊格式: "0.8-1元/月/㎡" → 平均值=(0.8+1)/2=0.9
    """
    if pd.isna(property_fee_str):
        return np.nan
    
    property_fee_str = str(property_fee_str).strip()
    
    try:
        # 方法1: 直接匹配数字和区间格式
        # 匹配单个数字: "2.8元/月/㎡"
        single_match = re.match(r'^(\d+\.?\d*)元/月/㎡$', property_fee_str)
        if single_match:
            return float(single_match.group(1))
        
        # 匹配区间格式: "2.38-2.98元/月/㎡"
        range_match = re.match(r'^(\d+\.?\d*)-(\d+\.?\d*)元/月/㎡$', property_fee_str)
        if range_match:
            fee_min = float(range_match.group(1))
            fee_max = float(range_match.group(2))
            return (fee_min + fee_max) / 2
        
        # 方法2: 提取所有数字并处理
        numbers = re.findall(r'\d+\.?\d*', property_fee_str)
        if len(numbers) == 1:
            # 只有一个数字
            return float(numbers[0])
        elif len(numbers) == 2:
            # 有两个数字，认为是区间
            fee_min = float(numbers[0])
            fee_max = float(numbers[1])
            return (fee_min + fee_max) / 2
        else:
            # 无法解析的格式
            return np.nan
            
    except (ValueError, IndexError, AttributeError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析物业费 '{property_fee_str}': {e}")
        return np.nan

# 应用转换函数
for idx in rent_price_df.index:
    property_fee_str = rent_price_df.loc[idx, '物业费']
    if pd.notna(property_fee_str):
        property_fee_avg = extract_property_fee(property_fee_str)
        rent_price_df.loc[idx, 'property_fee_avg'] = property_fee_avg

print(f"\n物业费变量统计:")
print(f"非缺失值数量: {rent_price_df['property_fee_avg'].notna().sum()}")
if rent_price_df['property_fee_avg'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['property_fee_avg'].mean():.2f}")
    print(f"标准差: {rent_price_df['property_fee_avg'].std():.2f}")
    print(f"最小值: {rent_price_df['property_fee_avg'].min():.2f}")
    print(f"最大值: {rent_price_df['property_fee_avg'].max():.2f}")
    print(f"中位数: {rent_price_df['property_fee_avg'].median():.2f}")
    
    # 显示分布情况
    print(f"\n物业费分布概况:")
    fee_stats = rent_price_df['property_fee_avg'].describe()
    print(fee_stats)
    
    # 按区间显示分布
    bins = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, float('inf')]
    labels = ['0-1', '1-2', '2-3', '3-4', '4-5', '5-10', '10-20', '20-50', '50-100', '100+']
    fee_ranges = pd.cut(rent_price_df['property_fee_avg'], bins=bins, labels=labels, right=False)
    print(f"\n物业费区间分布:")
    print(fee_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['property_fee_avg'].isna().sum()}")

print("物业费变量创建完成（保留缺失值）")

物业费分布:
物业费
2.8元/月/㎡          916
0.8元/月/㎡          762
1元/月/㎡            730
1.2元/月/㎡          644
1.8元/月/㎡          584
                 ... 
4.8-18元/月/㎡         1
0.45-1.28元/月/㎡      1
0.85-1元/月/㎡         1
0.55-1.29元/月/㎡      1
0.6-20元/月/㎡         1
Name: count, Length: 989, dtype: int64

物业费缺失值数量: 8489

物业费变量统计:
非缺失值数量: 25528
平均值: 2.41
标准差: 4.10
最小值: 0.20
最大值: 76.45
中位数: 1.76

物业费分布概况:
count    25528.000000
mean         2.408548
std          4.101584
min          0.200000
25%          1.100000
50%          1.760000
75%          2.680000
max         76.450000
Name: property_fee_avg, dtype: float64

物业费区间分布:
property_fee_avg
0-1       4794
1-2       9841
2-3       6729
3-4       2515
4-5        507
5-10       638
10-20      317
20-50      100
50-100      87
100+         0
Name: count, dtype: int64
缺失值数量: 8489
物业费变量创建完成（保留缺失值）


In [21]:
# 2.19 供水
# 2.19.1 分布检查
print("供水分布:")
print(rent_price_df['供水'].value_counts())
# 检查缺失值情况
print(f"\n供水缺失值数量: {rent_price_df['供水'].isna().sum()}")

# 2.19.2 创建供水哑变量
# 删除已存在的供水哑变量列
water_supply_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('water_')]
rent_price_df = rent_price_df.drop(columns=water_supply_columns_to_drop)

# 创建供水类型到英文的映射字典
water_supply_mapping = {
    '民水': 'civil_water',
    '商水': 'commercial_water', 
    '商水/民水': 'mixed_water'
}

# 将供水类型转换为英文
rent_price_df['water_type_en'] = rent_price_df['供水'].map(water_supply_mapping)

# 创建民水和商水哑变量
rent_price_df['water_civil'] = 0
rent_price_df['water_commercial'] = 0

# 根据供水类型设置哑变量值
rent_price_df.loc[rent_price_df['water_type_en'] == 'civil_water', 'water_civil'] = 1
rent_price_df.loc[rent_price_df['water_type_en'] == 'commercial_water', 'water_commercial'] = 1
rent_price_df.loc[rent_price_df['water_type_en'] == 'mixed_water', 'water_civil'] = 1
rent_price_df.loc[rent_price_df['water_type_en'] == 'mixed_water', 'water_commercial'] = 1

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['供水'].isna().any():
    na_mask = rent_price_df['供水'].isna()
    rent_price_df.loc[na_mask, 'water_civil'] = np.nan
    rent_price_df.loc[na_mask, 'water_commercial'] = np.nan

# 删除中间列
rent_price_df = rent_price_df.drop('water_type_en', axis=1)

print(f"\n已创建的供水哑变量:")
water_dummy_columns = ['water_civil', 'water_commercial']
print(water_dummy_columns)

print("\n供水哑变量分布:")
for col in water_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("供水哑变量创建完成（保留缺失值）")

供水分布:
供水
民水       17886
商水/民水     7777
商水         159
Name: count, dtype: int64

供水缺失值数量: 8195

已创建的供水哑变量:
['water_civil', 'water_commercial']

供水哑变量分布:
water_civil: 25663个为1, 159个为0, 8195个缺失
water_commercial: 7936个为1, 17886个为0, 8195个缺失
供水哑变量创建完成（保留缺失值）


In [22]:
# 2.19 建筑结构_comm
# 2.19.1 分布检查
print("建筑结构_comm分布:")
print(rent_price_df['建筑结构_comm'].value_counts())

# 2.19.2 缺失值检查
print(f"\n建筑结构_comm缺失值数量: {rent_price_df['建筑结构_comm'].isna().sum()}")

# 2.19.3 创建建筑结构_comm哑变量
# 删除已存在的建筑结构_comm哑变量列（使用特定前缀）
structure_type_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('structure_type_')]
rent_price_df = rent_price_df.drop(columns=structure_type_columns_to_drop)

# 创建四个主要的建筑结构_comm哑变量
rent_price_df['structure_type_tower'] = 0  # 塔楼
rent_price_df['structure_type_slab'] = 0   # 板楼  
rent_price_df['structure_type_combined'] = 0  # 塔板结合
rent_price_df['structure_type_bungalow'] = 0  # 平房

# 根据建筑结构_comm内容设置哑变量
for idx in rent_price_df.index:
    structure = rent_price_df.loc[idx, '建筑结构_comm']
    if pd.notna(structure):
        structure_str = str(structure).strip()
        
        # 设置塔楼哑变量
        if '塔楼' in structure_str:
            rent_price_df.loc[idx, 'structure_type_tower'] = 1
            
        # 设置板楼哑变量  
        if '板楼' in structure_str:
            rent_price_df.loc[idx, 'structure_type_slab'] = 1
            
        # 设置塔板结合哑变量
        if '塔板结合' in structure_str:
            rent_price_df.loc[idx, 'structure_type_combined'] = 1
            
        # 设置平房哑变量
        if '平房' in structure_str:
            rent_price_df.loc[idx, 'structure_type_bungalow'] = 1

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['建筑结构_comm'].isna().any():
    na_mask = rent_price_df['建筑结构_comm'].isna()
    rent_price_df.loc[na_mask, 'structure_type_tower'] = np.nan
    rent_price_df.loc[na_mask, 'structure_type_slab'] = np.nan
    rent_price_df.loc[na_mask, 'structure_type_combined'] = np.nan
    rent_price_df.loc[na_mask, 'structure_type_bungalow'] = np.nan

print(f"\n已创建的建筑结构(类型)哑变量:")
structure_type_dummy_columns = ['structure_type_tower', 'structure_type_slab', 'structure_type_combined', 'structure_type_bungalow']
print(structure_type_dummy_columns)

print("\n建筑结构(类型)哑变量分布:")
for col in structure_type_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("建筑结构(类型)哑变量创建完成（保留缺失值）")

建筑结构_comm分布:
建筑结构_comm
板楼               8770
塔楼               4468
塔楼/板楼            3170
塔楼/板楼/塔板结合       3146
板楼/塔板结合          2461
塔板结合             1787
塔楼/塔板结合          1125
塔楼/板楼/塔板结合/平房     467
塔楼/板楼/平房          237
板楼/平房             209
板楼/塔板结合/平房        134
塔楼/平房             127
塔板结合/平房            53
平房                 27
塔楼/塔板结合/平房          2
Name: count, dtype: int64

建筑结构_comm缺失值数量: 7834

已创建的建筑结构(类型)哑变量:
['structure_type_tower', 'structure_type_slab', 'structure_type_combined', 'structure_type_bungalow']

建筑结构(类型)哑变量分布:
structure_type_tower: 12742个为1, 13441个为0, 7834个缺失
structure_type_slab: 18594个为1, 7589个为0, 7834个缺失
structure_type_combined: 9175个为1, 17008个为0, 7834个缺失
structure_type_bungalow: 1256个为1, 24927个为0, 7834个缺失
建筑结构(类型)哑变量创建完成（保留缺失值）


In [23]:
# 2.20 供暖
# 2.20.1 分布检查
print("供暖分布:")
print(rent_price_df['供暖'].value_counts())
# 检查缺失值情况
print(f"\n供暖缺失值数量: {rent_price_df['供暖'].isna().sum()}")

# 2.20.2 创建供暖哑变量
# 删除已存在的供暖哑变量列
heating_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('heating_')]
rent_price_df = rent_price_df.drop(columns=heating_columns_to_drop)

# 创建供暖类型到英文的映射字典
heating_mapping = {
    '集中供暖': 'central_heating',
    '自采暖': 'self_heating', 
    '集中供暖/自采暖': 'mixed_heating',
    '无供暖': 'no_heating',
    '自采暖/无供暖': 'no_heating',
    '集中供暖/自采暖/无供暖': 'no_heating'
}

# 将供暖类型转换为英文
rent_price_df['heating_type_en'] = rent_price_df['供暖'].map(heating_mapping)

# 创建集中供暖和自采暖哑变量
rent_price_df['heating_central'] = 0
rent_price_df['heating_self'] = 0

# 根据供暖类型设置哑变量值
rent_price_df.loc[rent_price_df['heating_type_en'] == 'central_heating', 'heating_central'] = 1
rent_price_df.loc[rent_price_df['heating_type_en'] == 'self_heating', 'heating_self'] = 1
rent_price_df.loc[rent_price_df['heating_type_en'] == 'mixed_heating', 'heating_central'] = 1
rent_price_df.loc[rent_price_df['heating_type_en'] == 'mixed_heating', 'heating_self'] = 1
# 无供暖情况两个哑变量都为0，已经默认设置为0

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['供暖'].isna().any():
    na_mask = rent_price_df['供暖'].isna()
    rent_price_df.loc[na_mask, 'heating_central'] = np.nan
    rent_price_df.loc[na_mask, 'heating_self'] = np.nan

# 删除中间列
rent_price_df = rent_price_df.drop('heating_type_en', axis=1)

print(f"\n已创建的供暖哑变量:")
heating_dummy_columns = ['heating_central', 'heating_self']
print(heating_dummy_columns)

print("\n供暖哑变量分布:")
for col in heating_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("供暖哑变量创建完成（保留缺失值）")

供暖分布:
供暖
集中供暖            7197
自采暖             6042
集中供暖/自采暖        1752
无供暖              202
集中供暖/自采暖/无供暖     146
自采暖/无供暖           76
Name: count, dtype: int64

供暖缺失值数量: 18602

已创建的供暖哑变量:
['heating_central', 'heating_self']

供暖哑变量分布:
heating_central: 8949个为1, 6466个为0, 18602个缺失
heating_self: 7794个为1, 7621个为0, 18602个缺失
供暖哑变量创建完成（保留缺失值）


In [24]:
# 2.21 供电
# 2.21.1 分布检查
print("供电分布:")
print(rent_price_df['供电'].value_counts())
# 检查缺失值情况
print(f"\n供电缺失值数量: {rent_price_df['供电'].isna().sum()}")

# 2.21.2 创建供电哑变量
# 删除已存在的供电哑变量列
electricity_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('electricity_')]
rent_price_df = rent_price_df.drop(columns=electricity_columns_to_drop)

# 创建供电类型到英文的映射字典
electricity_mapping = {
    '民电': 'civil_electricity',
    '商电': 'commercial_electricity', 
    '商电/民电': 'mixed_electricity'
}

# 将供电类型转换为英文
rent_price_df['electricity_type_en'] = rent_price_df['供电'].map(electricity_mapping)

# 创建民电和商电哑变量
rent_price_df['electricity_civil'] = 0
rent_price_df['electricity_commercial'] = 0

# 根据供电类型设置哑变量值
rent_price_df.loc[rent_price_df['electricity_type_en'] == 'civil_electricity', 'electricity_civil'] = 1
rent_price_df.loc[rent_price_df['electricity_type_en'] == 'commercial_electricity', 'electricity_commercial'] = 1
rent_price_df.loc[rent_price_df['electricity_type_en'] == 'mixed_electricity', 'electricity_civil'] = 1
rent_price_df.loc[rent_price_df['electricity_type_en'] == 'mixed_electricity', 'electricity_commercial'] = 1

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['供电'].isna().any():
    na_mask = rent_price_df['供电'].isna()
    rent_price_df.loc[na_mask, 'electricity_civil'] = np.nan
    rent_price_df.loc[na_mask, 'electricity_commercial'] = np.nan

# 删除中间列
rent_price_df = rent_price_df.drop('electricity_type_en', axis=1)

print(f"\n已创建的供电哑变量:")
electricity_dummy_columns = ['electricity_civil', 'electricity_commercial']
print(electricity_dummy_columns)

print("\n供电哑变量分布:")
for col in electricity_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("供电哑变量创建完成（保留缺失值）")

供电分布:
供电
民电       17522
商电/民电     8160
商电         158
Name: count, dtype: int64

供电缺失值数量: 8177

已创建的供电哑变量:
['electricity_civil', 'electricity_commercial']

供电哑变量分布:
electricity_civil: 25682个为1, 158个为0, 8177个缺失
electricity_commercial: 8318个为1, 17522个为0, 8177个缺失
供电哑变量创建完成（保留缺失值）


In [25]:
# 2.22 燃气费
# 2.22.1 分布检查
print("燃气费分布:")
print(rent_price_df['燃气费'].value_counts())

# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['燃气费'].value_counts())
# pd.reset_option('display.max_rows')

# 2.22.2 创建燃气费变量
# 删除已存在的燃气费变量列
gas_columns_to_drop = [col for col in rent_price_df.columns if 
                      col in ['gas_fee_avg']]
rent_price_df = rent_price_df.drop(columns=gas_columns_to_drop)

# 初始化燃气费变量为NaN
rent_price_df['gas_fee_avg'] = np.nan

def extract_gas_fee(gas_fee_str):
    """
    从燃气费字符串中提取平均值
    处理逻辑:
    - 单个数值: "2.61元/m³" → 平均值=2.61
    - 区间数值: "2.61-2.63元/m³" → 平均值=(2.61+2.63)/2=2.62
    - 单位统一为: 元/m³
    """
    if pd.isna(gas_fee_str):
        return np.nan
    
    gas_fee_str = str(gas_fee_str).strip()
    
    try:
        # 处理单个数值格式
        if '元/m³' in gas_fee_str and '-' not in gas_fee_str:
            # 提取数字部分
            fee_match = re.search(r'(\d+\.?\d*)', gas_fee_str)
            if fee_match:
                return float(fee_match.group(1))
        
        # 处理区间格式
        elif '-' in gas_fee_str and '元/m³' in gas_fee_str:
            # 提取两个数字
            fee_range = re.findall(r'(\d+\.?\d*)', gas_fee_str)
            if len(fee_range) == 2:
                fee_min = float(fee_range[0])
                fee_max = float(fee_range[1])
                # 计算平均值
                return (fee_min + fee_max) / 2
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError, AttributeError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析燃气费 '{gas_fee_str}': {e}")
        return np.nan

# 应用转换函数
for idx in rent_price_df.index:
    gas_fee_str = rent_price_df.loc[idx, '燃气费']
    if pd.notna(gas_fee_str):
        gas_fee_avg = extract_gas_fee(gas_fee_str)
        rent_price_df.loc[idx, 'gas_fee_avg'] = gas_fee_avg

print(f"\n燃气费变量统计:")
print(f"非缺失值数量: {rent_price_df['gas_fee_avg'].notna().sum()}")
if rent_price_df['gas_fee_avg'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['gas_fee_avg'].mean():.2f}")
    print(f"标准差: {rent_price_df['gas_fee_avg'].std():.2f}")
    print(f"最小值: {rent_price_df['gas_fee_avg'].min():.2f}")
    print(f"最大值: {rent_price_df['gas_fee_avg'].max():.2f}")
    print(f"中位数: {rent_price_df['gas_fee_avg'].median():.2f}")
    
    # 显示分布情况
    print(f"\n燃气费分布概况:")
    fee_stats = rent_price_df['gas_fee_avg'].describe()
    print(fee_stats)
    
    # 按区间显示分布
    bins = [0, 1, 2, 2.5, 3, 3.5, 4, 5, 10, float('inf')]
    labels = ['0-1', '1-2', '2-2.5', '2.5-3', '3-3.5', '3.5-4', '4-5', '5-10', '10+']
    fee_ranges = pd.cut(rent_price_df['gas_fee_avg'], bins=bins, labels=labels, right=False)
    print(f"\n燃气费区间分布:")
    print(fee_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['gas_fee_avg'].isna().sum()}")

print("燃气费变量创建完成")

燃气费分布:
燃气费
2.61元/m³         6105
3元/m³            4056
3.45元/m³         3074
1.98元/m³         1815
2.61-2.63元/m³     794
                 ... 
2.9-3.95元/m³        1
2.9-2.97元/m³        1
2.5-3.31元/m³        1
2.9-3.99元/m³        1
2.05-2.4元/m³        1
Name: count, Length: 238, dtype: int64

燃气费变量统计:
非缺失值数量: 24860
平均值: 2.77
标准差: 0.55
最小值: 0.40
最大值: 5.00
中位数: 2.61

燃气费分布概况:
count    24860.000000
mean         2.767434
std          0.548511
min          0.400000
25%          2.470000
50%          2.610000
75%          3.000000
max          5.000000
Name: gas_fee_avg, dtype: float64

燃气费区间分布:
gas_fee_avg
0-1        12
1-2      4054
2-2.5    2246
2.5-3    8418
3-3.5    8417
3.5-4    1314
4-5       394
5-10        5
10+         0
Name: count, dtype: int64
缺失值数量: 9157
燃气费变量创建完成


In [26]:
# 2.23 供热费
# 2.23.1 分布检查
print("供热费分布:")
print(rent_price_df['供热费'].value_counts())
# 在初步调研时需要显示所有行，可以运行以下代码
# import pandas as pd
# pd.set_option('display.max_rows', None)
# print(rent_price_df['供热费'].value_counts())
# pd.reset_option('display.max_rows')

# 2.23.2 创建供热费变量
# 删除已存在的供热费变量列
heating_columns_to_drop = [col for col in rent_price_df.columns if 
                          col in ['heating_fee_avg']]
rent_price_df = rent_price_df.drop(columns=heating_columns_to_drop)

# 初始化供热费变量为NaN
rent_price_df['heating_fee_avg'] = np.nan

def extract_heating_fee(heating_fee_str):
    """
    从供热费字符串中提取平均值
    处理逻辑:
    - 单个数值: "30元/㎡" → 平均值=30
    - 区间数值: "25-35元/㎡" → 平均值=(25+35)/2=30
    - 单位统一为: 元/㎡
    """
    if pd.isna(heating_fee_str):
        return np.nan
    
    heating_fee_str = str(heating_fee_str).strip()
    
    try:
        # 处理单个数值格式
        if '元/㎡' in heating_fee_str and '-' not in heating_fee_str:
            # 提取数字部分
            fee_match = re.search(r'(\d+\.?\d*)', heating_fee_str)
            if fee_match:
                return float(fee_match.group(1))
        
        # 处理区间格式
        elif '-' in heating_fee_str and '元/㎡' in heating_fee_str:
            # 提取两个数字
            fee_range = re.findall(r'(\d+\.?\d*)', heating_fee_str)
            if len(fee_range) == 2:
                fee_min = float(fee_range[0])
                fee_max = float(fee_range[1])
                # 计算平均值
                return (fee_min + fee_max) / 2
        
        # 其他格式无法解析，返回NaN
        return np.nan
        
    except (ValueError, IndexError, AttributeError) as e:
        # 如果解析失败，返回NaN
        print(f"警告: 无法解析供热费 '{heating_fee_str}': {e}")
        return np.nan

# 应用转换函数
for idx in rent_price_df.index:
    heating_fee_str = rent_price_df.loc[idx, '供热费']
    if pd.notna(heating_fee_str):
        heating_fee_avg = extract_heating_fee(heating_fee_str)
        rent_price_df.loc[idx, 'heating_fee_avg'] = heating_fee_avg

print(f"\n供热费变量统计:")
print(f"非缺失值数量: {rent_price_df['heating_fee_avg'].notna().sum()}")
if rent_price_df['heating_fee_avg'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['heating_fee_avg'].mean():.2f}")
    print(f"标准差: {rent_price_df['heating_fee_avg'].std():.2f}")
    print(f"最小值: {rent_price_df['heating_fee_avg'].min():.2f}")
    print(f"最大值: {rent_price_df['heating_fee_avg'].max():.2f}")
    print(f"中位数: {rent_price_df['heating_fee_avg'].median():.2f}")
    
    # 显示分布情况
    print(f"\n供热费分布概况:")
    fee_stats = rent_price_df['heating_fee_avg'].describe()
    print(fee_stats)
    
    # 按区间显示分布
    bins = [0, 1, 5, 10, 20, 25, 30, 35, 40, 50, float('inf')]
    labels = ['0-1', '1-5', '5-10', '10-20', '20-25', '25-30', '30-35', '35-40', '40-50', '50+']
    fee_ranges = pd.cut(rent_price_df['heating_fee_avg'], bins=bins, labels=labels, right=False)
    print(f"\n供热费区间分布:")
    print(fee_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['heating_fee_avg'].isna().sum()}")


print("供热费变量创建完成")

供热费分布:
供热费
30元/㎡       4300
1元/㎡        1109
24-30元/㎡     697
22元/㎡        583
24元/㎡        572
            ... 
7.3元/㎡         1
29元/㎡          1
3.59元/㎡        1
5.25元/㎡        1
5.96元/㎡        1
Name: count, Length: 127, dtype: int64

供热费变量统计:
非缺失值数量: 10989
平均值: 20.60
标准差: 12.28
最小值: 0.01
最大值: 50.00
中位数: 27.00

供热费分布概况:
count    10989.000000
mean        20.597541
std         12.279816
min          0.010000
25%          5.300000
50%         27.000000
75%         30.000000
max         50.000000
Name: heating_fee_avg, dtype: float64

供热费区间分布:
heating_fee_avg
0-1       414
1-5      2265
5-10      503
10-20     241
20-25    1470
25-30    1275
30-35    4449
35-40     179
40-50     187
50+         6
Name: count, dtype: int64
缺失值数量: 23028
供热费变量创建完成


In [27]:
# 2.24 停车位
# 2.24.1 分布检查
print("停车位分布:")
print(rent_price_df['停车位'].value_counts())

# 2.24.2 缺失值检查
print(f"\n停车位缺失值数量: {rent_price_df['停车位'].isna().sum()}")

# 2.24.3 创建停车位变量
# 删除已存在的停车位变量列
parking_columns_to_drop = [col for col in rent_price_df.columns if 
                          col in ['parking_spots']]
rent_price_df = rent_price_df.drop(columns=parking_columns_to_drop)

# 初始化停车位变量，直接使用原数据
rent_price_df['parking_spots'] = rent_price_df['停车位']

# 确保停车位变量为数值类型
rent_price_df['parking_spots'] = pd.to_numeric(rent_price_df['parking_spots'], errors='coerce')

print(f"\n停车位变量统计:")
print(f"非缺失值数量: {rent_price_df['parking_spots'].notna().sum()}")
if rent_price_df['parking_spots'].notna().sum() > 0:
    print(f"平均值: {rent_price_df['parking_spots'].mean():.2f}")
    print(f"标准差: {rent_price_df['parking_spots'].std():.2f}")
    print(f"最小值: {rent_price_df['parking_spots'].min():.2f}")
    print(f"最大值: {rent_price_df['parking_spots'].max():.2f}")
    print(f"中位数: {rent_price_df['parking_spots'].median():.2f}")
    
    # 显示分布情况
    print(f"\n停车位分布概况:")
    spots_stats = rent_price_df['parking_spots'].describe()
    print(spots_stats)
    
    # 按区间显示分布
    bins = [0, 1, 10, 50, 100, 200, 500, 1000, float('inf')]
    labels = ['0-1', '1-10', '10-50', '50-100', '100-200', '200-500', '500-1000', '1000+']
    spots_ranges = pd.cut(rent_price_df['parking_spots'], bins=bins, labels=labels, right=False)
    print(f"\n停车位区间分布:")
    print(spots_ranges.value_counts().sort_index())

print(f"缺失值数量: {rent_price_df['parking_spots'].isna().sum()}")

print("停车位变量创建完成")

停车位分布:
停车位
500.0     1072
200.0      854
300.0      776
100.0      643
1000.0     555
          ... 
1129.0       1
16.0         1
162.0        1
6.0          1
257.0        1
Name: count, Length: 793, dtype: int64

停车位缺失值数量: 9571

停车位变量统计:
非缺失值数量: 24446
平均值: 1114.22
标准差: 1388.95
最小值: 1.00
最大值: 8700.00
中位数: 650.00

停车位分布概况:
count    24446.000000
mean      1114.219831
std       1388.954171
min          1.000000
25%        255.000000
50%        650.000000
75%       1495.000000
max       8700.000000
Name: parking_spots, dtype: float64

停车位区间分布:
parking_spots
0-1            0
1-10         395
10-50        623
50-100       931
100-200     2470
200-500     5214
500-1000    5081
1000+       9732
Name: count, dtype: int64
缺失值数量: 9571
停车位变量创建完成


In [28]:
# 2.25 周边配套
# 2.25.1 分布检查
print("周边配套分布:")
print(rent_price_df['周边配套'].value_counts().head(10))

# 2.25.2 异常值处理
# 检查是否有异常值或缺失值
print(f"\n周边配套缺失值数量: {rent_price_df['周边配套'].isna().sum()}")

# 2.25.3 创建周边配套哑变量
# 删除已存在的周边配套哑变量列
surrounding_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('surrounding_')]
rent_price_df = rent_price_df.drop(columns=surrounding_columns_to_drop)

# 创建周边配套哑变量函数
def check_surrounding_facilities(text):
    """
    检查周边配套中是否包含特定设施
    返回包含的设施列表
    """
    facilities = []
    if pd.isna(text):
        return facilities
    
    text_str = str(text)
    
    # 使用正则表达式检查医院
    hospital_pattern = r'医院|医疗|诊所|卫生院|三甲|二甲|人民医院'
    if re.search(hospital_pattern, text_str):
        facilities.append('hospital')
    
    # 使用正则表达式检查大学
    university_pattern = r'大学|学院|高校|高等院校|高职|大专'
    if re.search(university_pattern, text_str):
        facilities.append('university')
    
    # 使用正则表达式检查学校（中小学）
    school_pattern = r'小学|中学|初中|高中|学校|实验学校|外国语学校'
    if re.search(school_pattern, text_str):
        facilities.append('school')
    
    # 使用正则表达式检查超市
    supermarket_pattern = r'超市|商场|购物中心|百货|卖场|家乐福|沃尔玛|永辉|华润万家'
    if re.search(supermarket_pattern, text_str):
        facilities.append('supermarket')
    
    return facilities

# 应用函数并创建哑变量
import re

# 获取所有周边配套的设施信息
surrounding_facilities = rent_price_df['周边配套'].apply(check_surrounding_facilities)

# 创建各个设施的哑变量
rent_price_df['surrounding_hospital'] = surrounding_facilities.apply(lambda x: 1 if 'hospital' in x else 0)
rent_price_df['surrounding_university'] = surrounding_facilities.apply(lambda x: 1 if 'university' in x else 0)
rent_price_df['surrounding_school'] = surrounding_facilities.apply(lambda x: 1 if 'school' in x else 0)
rent_price_df['surrounding_supermarket'] = surrounding_facilities.apply(lambda x: 1 if 'supermarket' in x else 0)

print(f"\n已创建的周边配套哑变量:")
print(f"surrounding_hospital: 1表示附近有医院，0表示没有")
print(f"surrounding_university: 1表示附近有大学，0表示没有")
print(f"surrounding_school: 1表示附近有学校，0表示没有")
print(f"surrounding_supermarket: 1表示附近有超市，0表示没有")

print("\n周边配套哑变量分布:")
surrounding_dummy_columns = ['surrounding_hospital', 'surrounding_university', 'surrounding_school', 'surrounding_supermarket']
for col in surrounding_dummy_columns:
    true_count = (rent_price_df[col] == 1).sum()
    false_count = (rent_price_df[col] == 0).sum()
    print(f"{col}: {true_count}个为1, {false_count}个为0")

print("周边配套哑变量创建完成")

周边配套分布:
周边配套
一个地铁线：11号线昌吉东路站2.两大医院：安亭医院、东方肝胆医院3.一大公园：上海汽车博览公园4.三大商场：嘉亭荟大润发广场、底特律财富广场、三德广场                                          53
南山维拉项目北临甪直大道，西临长虹北路，南临迎宾西路。项目是由南山地产开发的70年产权住宅小区，共由13幢电梯洋房和19幢高层、约3万方维乐城、幼儿园组成；整体占地面积为17万余平米，建筑面积近50万平米                45
农业银行、工商银行、中国银行、招商银行、建设银行广州中医药大学附属骨伤科门诊部、黄岐医院、金铂天地、万达广场、万益广场、好又多、宏城、百佳、嘉和、卜峰莲花                                         40
自2012年启动建设以来，已经有包括古滇艺海大码，头朵拉萌宠乐园，古滇温泉酒店山庄，古滇精品湿地公园和七彩云南·欢乐世界主题公园在内的多个配套设施开放运营                                         38
小区自带底商，步行1.6公里有金地集中5万方购物广场，可以实现吃饭，购物，逛街，看电影一站式，步行1.1公里重庆西区医院（二甲能够为您和您的家人的健康，进行保驾护航，邻近有5所公园，都是免费对外开放，是您饭后茶余，周末的好去处。    35
1.小区北面就是占地81万方的上海植物园2.龙川北路百色路路口在建西南医疗医院3.小区西门就可以买菜，肉类日，海鲜，蔬菜都有或者可以到嘉陵菜场4．百色路有建设银行，平安银行，工商银行。                          29
大润发、商店、菜场、中国银行、农商银行、邮政储蓄银行、工商银行、海鲜、烧烤、红高粱、足浴、健身房等均在步行500米左右的范围内                                                       28
位于南滨路滨江沿线，它北临长江，背依南山，楼盘位置处可看到渝中夜景，旁边有巴渝十二景中的海棠烟雨、长嘉汇购物公园、弹子石老街、慈云寺、法国水师兵营、蓝光耍街和美心洋人街，吃喝玩乐、出行游玩都比较方便。                  27
小区外就是140亩公园+巴南万达广场+

In [29]:
# 2.26 建筑结构
# 2.26.1  分布检查
print("建筑结构分布:")
print(rent_price_df['建筑结构'].value_counts())
# 检查缺失值情况
print(f"\n建筑结构缺失值数量: {rent_price_df['建筑结构'].isna().sum()}")

# 2.26.2 创建建筑结构哑变量
# 删除已存在的建筑结构哑变量列（使用特定前缀）
structure_material_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('structure_material_')]
rent_price_df = rent_price_df.drop(columns=structure_material_columns_to_drop)

# 创建建筑结构到英文的映射字典
structure_material_mapping = {
    '钢混结构': 'steel_concrete',
    '混合结构': 'mixed', 
    '未知结构': 'unknown',
    '砖混结构': 'brick_concrete',
    '框架结构': 'frame',
    '钢结构': 'steel',
    '砖木结构': 'brick_wood'
}

# 将建筑结构转换为英文并创建哑变量
rent_price_df['structure_material_type_en'] = rent_price_df['建筑结构'].map(structure_material_mapping)
structure_material_dummies = pd.get_dummies(rent_price_df['structure_material_type_en'], prefix='structure_material')

# 将布尔值转换为整数 (True/False -> 1/0)
structure_material_dummies = structure_material_dummies.astype(int)

# 将原始数据中的缺失值反映到哑变量中
if rent_price_df['建筑结构'].isna().any():
    na_mask = rent_price_df['建筑结构'].isna()
    for col in structure_material_dummies.columns:
        structure_material_dummies.loc[na_mask, col] = np.nan

# 将哑变量合并到原数据框并删除中间列
rent_price_df = pd.concat([rent_price_df, structure_material_dummies], axis=1)
rent_price_df = rent_price_df.drop('structure_material_type_en', axis=1)

print(f"\n已创建的建筑结构(材料)哑变量:")
structure_material_dummy_columns = [col for col in rent_price_df.columns if col.startswith('structure_material_')]
print(structure_material_dummy_columns)

print("\n建筑结构(材料)哑变量分布:")
for col in structure_material_dummy_columns:
    valid_count = rent_price_df[col].notna().sum()
    true_count = (rent_price_df[col] == 1).sum()
    na_count = rent_price_df[col].isna().sum()
    print(f"{col}: {true_count}个为1, {valid_count-true_count}个为0, {na_count}个缺失")

print("建筑结构(材料)哑变量创建完成（保留缺失值）")

建筑结构分布:
建筑结构
钢混结构    25065
混合结构     3460
砖混结构     2715
未知结构     1846
框架结构      651
钢结构       230
砖木结构       36
Name: count, dtype: int64

建筑结构缺失值数量: 14

已创建的建筑结构(材料)哑变量:
['structure_material_brick_concrete', 'structure_material_brick_wood', 'structure_material_frame', 'structure_material_mixed', 'structure_material_steel', 'structure_material_steel_concrete', 'structure_material_unknown']

建筑结构(材料)哑变量分布:
structure_material_brick_concrete: 2715个为1, 31288个为0, 14个缺失
structure_material_brick_wood: 36个为1, 33967个为0, 14个缺失
structure_material_frame: 651个为1, 33352个为0, 14个缺失
structure_material_mixed: 3460个为1, 30543个为0, 14个缺失
structure_material_steel: 230个为1, 33773个为0, 14个缺失
structure_material_steel_concrete: 25065个为1, 8938个为0, 14个缺失
structure_material_unknown: 1846个为1, 32157个为0, 14个缺失
建筑结构(材料)哑变量创建完成（保留缺失值）


In [30]:
# 2.27 物业办公电话
# 2.27.1 分布检查
print("物业办公电话分布:")
print(rent_price_df['物业办公电话'].value_counts().head(10))

# 2.27.2 异常值处理
# 检查是否有异常值或缺失值
print(f"\n物业办公电话缺失值数量: {rent_price_df['物业办公电话'].isna().sum()}")

# 2.27.3 创建物业办公电话哑变量
# 删除已存在的物业电话哑变量列
property_phone_columns_to_drop = [col for col in rent_price_df.columns if col.startswith('property_phone_')]
rent_price_df = rent_price_df.drop(columns=property_phone_columns_to_drop)

# 创建物业办公电话哑变量
# 规则：包含数字记为1，空值、"无"或非数字内容记为0
def has_phone_number(value):
    if pd.isna(value):
        return 0
    value_str = str(value).strip()
    if value_str == '' or value_str == '无' or value_str == '暂无' or value_str == '没有':
        return 0
    # 检查是否包含数字
    if any(char.isdigit() for char in value_str):
        return 1
    else:
        return 0

# 应用函数创建哑变量
rent_price_df['property_phone_yes'] = rent_price_df['物业办公电话'].apply(has_phone_number)

print(f"\n已创建的物业办公电话哑变量:")
print(f"property_phone_yes: 1表示有物业办公电话，0表示无物业办公电话")

print("\n物业办公电话哑变量分布:")
print(f"无物业办公电话 (property_phone_yes=0): {(rent_price_df['property_phone_yes'] == 0).sum()}")
print(f"有物业办公电话 (property_phone_yes=1): {(rent_price_df['property_phone_yes'] == 1).sum()}")
print(f"缺失值: {rent_price_df['property_phone_yes'].isna().sum()}")

print("物业办公电话哑变量创建完成")

物业办公电话分布:
物业办公电话
无                                                                   1419
0316-3302980                                                         162
15801295270                                                          135
010-81777036,010-61754236                                            118
0316-5996892                                                         107
010-68635929,010-88685149,010-88685249,010-68683551,010-68688551     100
暂无                                                                    98
021-37565506                                                          85
023-86968333                                                          80
020-37924001                                                          76
Name: count, dtype: int64

物业办公电话缺失值数量: 22593

已创建的物业办公电话哑变量:
property_phone_yes: 1表示有物业办公电话，0表示无物业办公电话

物业办公电话哑变量分布:
无物业办公电话 (property_phone_yes=0): 24124
有物业办公电话 (property_phone_yes=1): 9893
缺失值: 0
物业办公电话哑变量创建完成


In [31]:
# 2.28 到市中心的距离
# Haversine公式计算两个经纬度点之间的距离（单位：公里）
def calculate_distance(lon1, lat1, lon2, lat2):
    # 将十进制度数转化为弧度
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine公式
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    r = 6371  # 地球平均半径，单位为公里
    return c * r

# 定义各城市中心点经纬度字典（保持不变）
city_centers = {
    0: (116.3333, 39.9333),    # 城市0：北京
    1: (116.3333, 39.9333),    # 城市1：北京
    2: (106.33, 29.35),        # 城市2：重庆
    3: None,                   # 城市3：需要从数据中计算中心点
    4: (121.48, 31.23),        # 城市4：上海
    5: (113.56, 22.28),        # 城市5：广州
    6: (114.86, 40.8),         # 城市6：张家口
    7: (114.07, 22.62),        # 城市7：深圳
    8: (102.7086, 25.036),     # 城市8：昆明
    9: None,                   # 城市9：需要从数据中计算中心点
    10: (113.2647, 23.1089),   # 城市10：佛山
    11: (115.4667, 38.8667)    # 城市11：保定
}

# 对于城市3和城市9，从数据中计算中心点（取该城市经纬度的平均值）
def calculate_city_center(df, city_id):
    city_data = df[df['城市'] == city_id]
    if len(city_data) == 0:
        return None
    center_lon = city_data['lon'].mean()
    center_lat = city_data['lat'].mean()
    return (center_lon, center_lat)

# 计算距市中心距离的函数
def calculate_city_center_distance(row, df, city_centers_dict):
    city_id = row['城市']
    center_point = city_centers_dict.get(city_id)
    
    # 如果中心点未定义，则从数据中计算
    if center_point is None:
        center_point = calculate_city_center(df, city_id)
        if center_point is None:
            return np.nan
    
    center_lon, center_lat = center_point
    distance = calculate_distance(row['lon'], row['lat'], center_lon, center_lat)
    return distance

# 为购房数据计算距离
print("开始为购房数据计算距市中心距离...")

# 复制城市中心点字典，避免修改原始字典
price_city_centers = city_centers.copy()

# 计算购房数据的距市中心距离
rent_price_df['距市中心距离_km'] = rent_price_df.apply(
    lambda row: calculate_city_center_distance(row, rent_price_df, price_city_centers), 
    axis=1
)

# 检查结果
print("\n=== 购房数据距市中心距离计算完成 ===")
print(f"成功计算了 {rent_price_df['距市中心距离_km'].notna().sum()} 条数据的距离")
print(f"缺失值数量: {rent_price_df['距市中心距离_km'].isna().sum()}")

# 显示各城市距离的统计信息
print("\n=== 购房数据各城市距市中心距离统计（公里）===")
distance_stats = rent_price_df.groupby('城市')['距市中心距离_km'].agg(['count', 'mean', 'min', 'max']).round(2)
print(distance_stats)

# 显示前几行数据，包含新计算的距离列
print("\n=== 购房数据包含距市中心距离的前5行数据 ===")
columns_to_show = ['城市', 'lon', 'lat', '距市中心距离_km']
# 根据实际列名调整
if 'Price' in rent_price_df.columns:
    columns_to_show.insert(3, 'Price')
elif 'price' in rent_price_df.columns:
    columns_to_show.insert(3, 'price')
print(rent_price_df[columns_to_show].head())

print("\n购房数据处理完成！")

开始为购房数据计算距市中心距离...

=== 购房数据距市中心距离计算完成 ===
成功计算了 34017 条数据的距离
缺失值数量: 0

=== 购房数据各城市距市中心距离统计（公里）===
    count    mean     min     max
城市                               
0    7057  148.34   96.30  218.19
1    1985  148.72   95.38  181.76
2    5431  183.25  139.53  406.75
3    3961   25.71    2.93   71.53
4    5602  144.65  101.79  198.02
5    1057  138.75  107.66  164.83
6     933  154.60  137.08  261.80
7     879  153.81  133.23  179.94
8    2051  153.53  116.01  180.91
9    1182    3.66    0.25   32.03
10   3694  162.05  143.68  232.61
11    185  200.07  142.15  232.84

=== 购房数据包含距市中心距离的前5行数据 ===
   城市         lon        lat   距市中心距离_km
0   0  117.389491  40.901030  139.905245
1   0  117.376625  40.767478  128.140888
2   0  117.631276  41.063635  166.856539
3   0  117.186216  41.163738  154.634870
4   0  117.400114  40.959679  145.512669

购房数据处理完成！


In [32]:
# 创建最终的数据集并保存到本地

# 列出所有需要的变量
final_variables = [
   'ID',
   'decoration_精装',
   'decoration_简装',
   'decoration_毛坯',
   'decoration_其他',
   'high_dummy',
   'middle_dummy',
   'low_dummy',
   'basement_dummy',
   'top_dummy',
   'bottom_dummy',
   'total_floor',
   'area',
   'room_count',
   'hall_count',
   'south_dummy',
   'north_south_dummy',
   'trans_2024', 
   'trans_2025',
   '梯数',
   '户数',
   'elevator_yes',
   'transaction_commercial', 
   'transaction_non_commercial',
   'usage_commercial_office', 
   'usage_commercial_residential',
   'usage_high_end_residential', 
   'usage_ordinary_residential', 
   'usage_other_special',
   'house_age_over_2_years', 
   'house_age_over_5_years', 
   'house_age_under_2_years',
   'subway',
   'building_age',
   'household_total',
   'building_total',
   'greening_rate',
   'plot_ratio',
   'structure_material_brick_concrete', 
   'structure_material_brick_wood', 
   'structure_material_frame', 
   'structure_material_mixed', 
   'structure_material_steel', 
   'structure_material_steel_concrete', 
   'structure_material_unknown',
   'property_fee_avg',
   'water_civil',
   'water_commercial',
   'heating_central',
   'heating_self',
   'electricity_civil',
   'electricity_commercial',
   'gas_fee_avg',
   'heating_fee_avg',
   'parking_spots',
   'surrounding_hospital',
   'surrounding_university',
   'surrounding_school',
   'surrounding_supermarket',
   'property_phone_yes', 
   '城市',  # 添加城市列用于分组回归
   '区县',
   '板块',
   '距市中心距离_km'
]

# 检查哪些变量在数据集中存在
existing_variables = [var for var in final_variables if var in rent_price_df.columns]
missing_variables = [var for var in final_variables if var not in rent_price_df.columns]

print("数据集变量检查:")
print(f"存在的变量数量: {len(existing_variables)}")
print(f"缺失的变量数量: {len(missing_variables)}")

if missing_variables:
    print("\n缺失的变量:")
    for var in missing_variables:
        print(f"  - {var}")

# 创建最终数据集
final_df = rent_price_df[existing_variables].copy()

print(f"\n最终数据集形状: {final_df.shape}")
print(f"样本数量: {final_df.shape[0]}")
print(f"变量数量: {final_df.shape[1]}")

# 检查缺失值情况
print("\n各变量缺失值统计:")
missing_stats = final_df.isnull().sum()
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)

if len(missing_stats) > 0:
    print("有缺失值的变量:")
    for var, count in missing_stats.items():
        percentage = (count / len(final_df)) * 100
        print(f"  {var}: {count}个缺失值 ({percentage:.2f}%)")
else:
    print("没有缺失值")

# 保存到本地
output_file = 'house_price_final_test_dataset.csv'
final_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n最终数据集已保存到: {output_file}")

# 显示数据集的基本信息
print("\n数据集基本信息:")
print(final_df.info())

# 显示前几行数据
print("\n数据集前5行:")
print(final_df.head())

# 显示城市分布（用于分组回归）
print("\n城市分布（用于分组回归）:")
print(final_df['城市'].value_counts())

数据集变量检查:
存在的变量数量: 64
缺失的变量数量: 0

最终数据集形状: (34017, 64)
样本数量: 34017
变量数量: 64

各变量缺失值统计:
有缺失值的变量:
  heating_fee_avg: 23028个缺失值 (67.70%)
  heating_central: 18602个缺失值 (54.68%)
  heating_self: 18602个缺失值 (54.68%)
  house_age_over_5_years: 11296个缺失值 (33.21%)
  house_age_over_2_years: 11296个缺失值 (33.21%)
  house_age_under_2_years: 11296个缺失值 (33.21%)
  parking_spots: 9571个缺失值 (28.14%)
  building_age: 9406个缺失值 (27.65%)
  plot_ratio: 9303个缺失值 (27.35%)
  greening_rate: 9180个缺失值 (26.99%)
  gas_fee_avg: 9157个缺失值 (26.92%)
  property_fee_avg: 8489个缺失值 (24.96%)
  water_civil: 8195个缺失值 (24.09%)
  water_commercial: 8195个缺失值 (24.09%)
  electricity_civil: 8177个缺失值 (24.04%)
  electricity_commercial: 8177个缺失值 (24.04%)
  subway: 6379个缺失值 (18.75%)
  elevator_yes: 4092个缺失值 (12.03%)
  区县: 3732个缺失值 (10.97%)
  building_total: 3715个缺失值 (10.92%)
  household_total: 3715个缺失值 (10.92%)
  户数: 635个缺失值 (1.87%)
  梯数: 635个缺失值 (1.87%)
  hall_count: 164个缺失值 (0.48%)
  decoration_其他: 14个缺失值 (0.04%)
  decoration_毛坯: 14个缺失值 (0.04%)
